# Semantic Backup Validation -- Colab

Validates whether the semantic-similarity backup signal actually discriminates
between competencies, using **Salesforce/codet5p-110m-embedding** -- a code
embedding model trained with a contrastive objective specifically for code
similarity (replacing an earlier CodeBERT-based attempt, which was measured
in this same notebook to have no real discriminative power under either
mean- or [CLS]-pooling: SAME/CROSS/UNRELATED similarities all landed in a
narrow 0.96-0.99 band, worse than even the local lexical baseline). This
model needs network access to huggingface.co, which is why this runs in
Colab rather than the original blocked-network dev environment.

The exemplar registry now covers 10 competencies, mixing hand-written
snippets (all verified against the real structural resolver -- see Step 3b)
with exemplars mined directly from the real Refactory corpus via
`mine_exemplars.py`: `ConditionalCount`, `AccumulateThenCount`,
`FunctionalUniqueCheck` (the third real UniqueCheck strategy: map()/filter()
pipelines), and `DelegatesUniqueness` (a cross-function competency -- does
`contains_unique_day` correctly delegate to a `unique_day` that itself
satisfies `ConditionalCount`). Mining revealed `AccumulateThenCount` is a
genuinely rare real strategy (<1% of correct question_2 submissions) with
almost no real structural diversity to mine, unlike `ConditionalCount` (66
distinct real shapes) and `DelegatesUniqueness` (96 distinct real shapes,
78% of submissions).

**Before running:** have these five files from your project ready to upload
when prompted in Step 2: `competency.py`, `report.py`, `test_real_data.py`,
`pw01_competencies.py`, `q2_competencies.py`. (`semantic_backup.py` and
`validate_semantic_backup.py` are written for you in Step 3 below, so you
don't need to upload those.)


## 1. Install dependencies

In [1]:
!pip install -q transformers scikit-learn
# torch is preinstalled on Colab runtimes; this is a no-op if so, and
# installs it otherwise.
import importlib
if importlib.util.find_spec("torch") is None:
    !pip install -q torch


## 2. Write the project's structural model files
These are embedded directly below (verified in this session against the real Refactory corpus -- `audit_registry()` clean, `mine_exemplars.py` reproduces exactly) rather than requiring manual upload, so this notebook always runs against a known-good, matching set of files instead of whatever happens to be on hand locally.

Includes this round's structural engine updates: `TEST` now also fires on `while` conditions (not just `if`) and carries an `all_names` binding (every identifier referenced anywhere in the test, not just the left-hand operand); `SHARED_BINDING` now supports containment against a collection-valued binding; a new `SELF_RECURSIVE` primitive detects direct recursion. Plus the new `TopKPatternExtended` competency family (`TopKViaDelegate`, `HasSubstantialWork`, `SortThenSlice`) from the embedding-guided clustering work in Section 5.3 of the paper. Confirmed in this session: none of this changes the semantic backup layer's exemplar registry or mining numbers (`audit_registry()` still clean, `mine_exemplars.py` output is byte-identical) -- these are structural-layer improvements, orthogonal to Section 7's validation.

In [2]:
import io

competency_source = r'''
"""
Competency extraction model.

Two layers of competencies over a Python AST:
  - Primitive competencies: DECL, INPUT, OUTPUT, TEST, LOOP, ASSIGN
  - Advanced competencies: composed of primitives and/or other Advanced
    competencies, via a CoverageRule (how much must be present) and a
    StructuralConstraint (how the pieces must relate).

Every resolved competency (primitive or advanced) produces an Evidence
object exposing a line-span and role bindings, so Advanced competencies
can be nested inside further Advanced competencies uniformly.
"""

from __future__ import annotations
import ast
from dataclasses import dataclass, field
from typing import Callable, Optional


# ---------------------------------------------------------------------------
# Evidence
# ---------------------------------------------------------------------------

@dataclass
class Evidence:
    """A single match for a competency, primitive or composite."""
    competency_id: str
    node: Optional[ast.AST]          # underlying AST node (None for pure composites)
    start_line: int
    end_line: int
    bindings: dict = field(default_factory=dict)   # role -> variable/expr name (str)
    enclosing: Optional[ast.AST] = None             # nearest enclosing block, for NESTED checks
    scope: Optional[ast.AST] = None                  # enclosing FunctionDef (None = module-level); guards against cross-function contamination
    score: float = 1.0                               # own resolved score (matters for composites)
    children: dict = field(default_factory=dict)     # for composites: component_id -> Evidence

    def contains_line(self, line: int) -> bool:
        return self.start_line <= line <= self.end_line


@dataclass
class Result:
    """Outcome of resolving one competency against one submission."""
    competency_id: str
    coverage_score: float
    structural_score: float
    final_score: float
    evidences: list          # list[Evidence] chosen as the best match(es)
    detail: str = ""         # human-readable explanation, for feedback


# ---------------------------------------------------------------------------
# Helpers for reading source text and enclosing blocks
# ---------------------------------------------------------------------------

def _expr_str(node: Optional[ast.AST]) -> str:
    if node is None:
        return ""
    try:
        return ast.unparse(node).strip()
    except Exception:
        return ""


def _line_span(node: ast.AST):
    start = node.lineno
    end = getattr(node, "end_lineno", None) or start
    for child in ast.walk(node):
        cl = getattr(child, "end_lineno", None)
        if cl:
            end = max(end, cl)
    return start, end


class _ParentIndex:
    """Maps each node to its parent, so we can find 'enclosing block'."""
    def __init__(self, tree: ast.AST):
        self.parent = {}
        for node in ast.walk(tree):
            for child in ast.iter_child_nodes(node):
                self.parent[child] = node

    def enclosing_block(self, node: ast.AST):
        """Nearest ancestor that is a For/While/If/FunctionDef (a 'block')."""
        cur = self.parent.get(node)
        block_types = (ast.For, ast.While, ast.If, ast.FunctionDef)
        while cur is not None and not isinstance(cur, block_types):
            cur = self.parent.get(cur)
        return cur

    def enclosing_function(self, node: ast.AST):
        """Nearest ancestor FunctionDef, skipping past For/While/If -- this
        is 'which function's scope is this evidence in', distinct from
        enclosing_block which stops at the first block of any kind.
        Returns None for genuinely module-level evidence."""
        cur = self.parent.get(node)
        while cur is not None and not isinstance(cur, ast.FunctionDef):
            cur = self.parent.get(cur)
        return cur


# ---------------------------------------------------------------------------
# Primitive detectors
# Each takes (tree, parent_index) and returns a list[Evidence]
# ---------------------------------------------------------------------------

def detect_decl(tree, pidx) -> list:
    """Heuristic: first assignment to a name in a scope, with a simple
    literal/empty-collection value -> treated as a 'declaration'."""
    seen = set()
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and len(node.targets) == 1 \
                and isinstance(node.targets[0], ast.Name):
            name = node.targets[0].id
            if name in seen:
                continue
            is_simple_init = isinstance(node.value, (ast.Constant, ast.List, ast.Dict, ast.Set, ast.Tuple)) \
                or (isinstance(node.value, ast.Call) and _expr_str(node.value) in ("list()", "dict()", "set()"))
            if is_simple_init:
                seen.add(name)
                s, e = _line_span(node)
                out.append(Evidence("DECL", node, s, e,
                                     bindings={"target": name},
                                     enclosing=pidx.enclosing_block(node)))
    return out


def detect_input(tree, pidx) -> list:
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "input":
            # find the assignment this input() feeds, if any
            target = None
            p = pidx.parent.get(node)
            hops = 0
            while p is not None and hops < 4:
                if isinstance(p, ast.Assign) and len(p.targets) == 1 and isinstance(p.targets[0], ast.Name):
                    target = p.targets[0].id
                    break
                p = pidx.parent.get(p)
                hops += 1
            s, e = _line_span(node)
            out.append(Evidence("INPUT", node, s, e,
                                 bindings={"target": target},
                                 enclosing=pidx.enclosing_block(node)))
    return out


def detect_builtin_reduce(tree, pidx) -> list:
    """Calls to max()/min() -- the 'find the extremum via builtin' alternative
    to a manual TEST-based comparison loop. Whether a competency should
    credit this as equivalent to the manual form is a modeling choice, not
    something this detector decides; it just surfaces the evidence so a
    competency definition can choose ALL, ANY, or ignore it."""
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in ("max", "min"):
            s, e = _line_span(node)
            out.append(Evidence("BUILTIN_REDUCE", node, s, e,
                                 bindings={"op": node.func.id, "source": ", ".join(_expr_str(a) for a in node.args)},
                                 enclosing=pidx.enclosing_block(node)))
    return out


def detect_func_def(tree, pidx) -> list:
    """One evidence per function definition. Distinct from PARAM (which
    fires once per parameter): this is 'a function exists' as its own
    checkable fact, with whether it returns a value as a binding so
    'defined a function' and 'defined a function that returns something'
    can be told apart."""
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            has_return = any(isinstance(n, ast.Return) and n.value is not None for n in ast.walk(node))
            s, e = _line_span(node)
            out.append(Evidence("FUNC_DEF", node, s, e,
                                 bindings={"name": node.name,
                                           "n_params": str(len(node.args.args)),
                                           "has_return": "yes" if has_return else "no"},
                                 enclosing=node))
    return out


def detect_param(tree, pidx) -> list:
    """Function parameters are also a 'data enters here' primitive, distinct
    from INPUT (console input). Function-graded assignments (the common case
    in autograded exercises) receive data via parameters, not input()."""
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            for a in node.args.args:
                out.append(Evidence("PARAM", node, node.lineno, node.lineno,
                                     bindings={"target": a.arg},
                                     enclosing=node))
    return out


def detect_output(tree, pidx) -> list:
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Return) and node.value is not None:
            s, e = _line_span(node)
            out.append(Evidence("OUTPUT", node, s, e,
                                 bindings={"source": _expr_str(node.value)},
                                 enclosing=pidx.enclosing_block(node)))
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "print":
            s, e = _line_span(node)
            args = ", ".join(_expr_str(a) for a in node.args)
            out.append(Evidence("OUTPUT", node, s, e,
                                 bindings={"source": args},
                                 enclosing=pidx.enclosing_block(node)))
    return out


_CMP_OPS = {ast.Lt: "<", ast.LtE: "<=", ast.Gt: ">", ast.GtE: ">=", ast.Eq: "==", ast.NotEq: "!="}


def detect_test(tree, pidx) -> list:
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.If):
            operand = comparator = op = None
            test = node.test
            if isinstance(test, ast.Compare) and len(test.ops) == 1:
                operand = _expr_str(test.left)
                comparator = _expr_str(test.comparators[0])
                op = _CMP_OPS.get(type(test.ops[0]), "?")
            else:
                operand = _expr_str(test)
            # all_names: every bare identifier referenced anywhere in the
            # test expression, regardless of its shape (simple compare,
            # BoolOp `and`/`or`, or a chained compare like `a < x <= b`).
            # This is a strict superset of `operand` -- when the test IS a
            # simple single-op compare with the target on the left, operand
            # already equals the one relevant name, so anything that used to
            # match via `operand` still matches via `all_names`. What it
            # additionally catches: the target appearing on the right side
            # of a simple comparison, or anywhere inside a compound/chained
            # condition, which `operand`'s single flat string can't express.
            all_names = frozenset(n.id for n in ast.walk(test) if isinstance(n, ast.Name))
            s, e = _line_span(node)
            out.append(Evidence("TEST", node, s, e,
                                 bindings={"operand": operand, "comparator": comparator, "op": op,
                                           "all_names": all_names},
                                 enclosing=pidx.enclosing_block(node)))
        elif isinstance(node, ast.While):
            # A while loop's own condition functions exactly like a
            # per-iteration test (e.g. `while x > seq[count]:`), but it
            # isn't an ast.If, so it was previously invisible to TEST
            # entirely. Synthesized with node=None -- deliberately, not an
            # oversight -- so the resolver's node-reuse guard (which only
            # inspects non-None nodes) never treats this as colliding with
            # the LOOP evidence generated for the very same while statement;
            # composites use the same node=None convention for an analogous
            # reason. Because node=None, the generic post-resolve scope-
            # tagging pass (which skips node=None evidence) would never run
            # for this one, so scope is computed and set directly here,
            # from the real while node, instead of relying on that pass.
            test = node.test
            operand = comparator = op = None
            if isinstance(test, ast.Compare) and len(test.ops) == 1:
                operand = _expr_str(test.left)
                comparator = _expr_str(test.comparators[0])
                op = _CMP_OPS.get(type(test.ops[0]), "?")
            else:
                operand = _expr_str(test)
            all_names = frozenset(n.id for n in ast.walk(test) if isinstance(n, ast.Name))
            s, e = _line_span(node)
            out.append(Evidence("TEST", None, s, e,
                                 bindings={"operand": operand, "comparator": comparator, "op": op,
                                           "all_names": all_names},
                                 enclosing=pidx.enclosing_block(node),
                                 scope=pidx.enclosing_function(node)))
    return out


def detect_loop(tree, pidx) -> list:
    out = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.For, ast.While)):
            iterator_var = None
            bound_expr = None
            kind = "for" if isinstance(node, ast.For) else "while"
            if isinstance(node, ast.For) and isinstance(node.target, ast.Name):
                iterator_var = node.target.id
                bound_expr = _expr_str(node.iter)
            elif isinstance(node, ast.While):
                bound_expr = _expr_str(node.test)
            s, e = _line_span(node)
            out.append(Evidence("LOOP", node, s, e,
                                 bindings={"iterator_var": iterator_var, "bound_expr": bound_expr, "kind": kind},
                                 enclosing=pidx.enclosing_block(node)))
    return out


def detect_assign(tree, pidx) -> list:
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and len(node.targets) == 1:
            target = _expr_str(node.targets[0])
            source = _expr_str(node.value)
            s, e = _line_span(node)
            out.append(Evidence("ASSIGN", node, s, e,
                                 bindings={"target": target, "source": source},
                                 enclosing=pidx.enclosing_block(node)))
        elif isinstance(node, ast.AugAssign):
            target = _expr_str(node.target)
            op = type(node.op).__name__
            source = f"{target} {op} {_expr_str(node.value)}"
            s, e = _line_span(node)
            out.append(Evidence("ASSIGN", node, s, e,
                                 bindings={"target": target, "source": source},
                                 enclosing=pidx.enclosing_block(node)))
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) \
                and node.func.attr in ("append", "add", "pop", "remove", "insert"):
            # collection mutation counts as an assignment-like effect
            target = _expr_str(node.func.value)
            source = ", ".join(_expr_str(a) for a in node.args)
            s, e = _line_span(node)
            out.append(Evidence("ASSIGN", node, s, e,
                                 bindings={"target": target, "source": source, "kind": node.func.attr},
                                 enclosing=pidx.enclosing_block(node)))
    return out


def detect_functional_collect(tree, pidx) -> list:
    """Calls to map()/filter() -- the functional-pipeline alternative to an
    explicit loop for building a derived collection."""
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in ("map", "filter"):
            s, e = _line_span(node)
            args_text = ", ".join(_expr_str(a) for a in node.args)
            out.append(Evidence("FUNCTIONAL_COLLECT", node, s, e,
                                 bindings={"func": node.func.id, "source": args_text},
                                 enclosing=pidx.enclosing_block(node)))
    return out


def detect_count_check(tree, pidx) -> list:
    """Calls to .count(...) or len(...) -- the 'how many/is there exactly
    one' check that typically follows a functional collect step."""
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr == "count":
            s, e = _line_span(node)
            out.append(Evidence("COUNT_CHECK", node, s, e,
                                 bindings={"kind": "count", "target": _expr_str(node.func.value)},
                                 enclosing=pidx.enclosing_block(node)))
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "len":
            s, e = _line_span(node)
            target = _expr_str(node.args[0]) if node.args else ""
            out.append(Evidence("COUNT_CHECK", node, s, e,
                                 bindings={"kind": "len", "target": target},
                                 enclosing=pidx.enclosing_block(node)))
    return out


def detect_self_recursive(tree, pidx) -> list:
    """One evidence per function that calls itself somewhere in its own
    body -- direct recursion only (a calls a, not a calls b calls a).
    Exists specifically because LOOP alone misses genuinely substantial
    recursive work: a real submission's merge_sort had zero LOOP primitives
    at all, being pure recursion, which a loop-only 'does real work' check
    incorrectly treated as doing nothing."""
    out = []
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            calls_self = any(
                isinstance(n, ast.Call) and isinstance(n.func, ast.Name) and n.func.id == node.name
                for n in ast.walk(node)
            )
            if calls_self:
                s, e = _line_span(node)
                out.append(Evidence("SELF_RECURSIVE", node, s, e,
                                     bindings={"name": node.name},
                                     enclosing=pidx.enclosing_block(node)))
    return out


PRIMITIVE_DETECTORS: dict = {
    "DECL": detect_decl,
    "INPUT": detect_input,
    "PARAM": detect_param,
    "FUNC_DEF": detect_func_def,
    "OUTPUT": detect_output,
    "TEST": detect_test,
    "LOOP": detect_loop,
    "ASSIGN": detect_assign,
    "BUILTIN_REDUCE": detect_builtin_reduce,
    "FUNCTIONAL_COLLECT": detect_functional_collect,
    "COUNT_CHECK": detect_count_check,
    "SELF_RECURSIVE": detect_self_recursive,
}


# ---------------------------------------------------------------------------
# Coverage rules
# ---------------------------------------------------------------------------

class CoverageRule:
    def score(self, component_scores: dict) -> float:
        raise NotImplementedError


@dataclass
class ALL(CoverageRule):
    ids: list
    def score(self, component_scores):
        if not self.ids:
            return 1.0
        return sum(component_scores.get(i, 0.0) for i in self.ids) / len(self.ids)


@dataclass
class ANY(CoverageRule):
    ids: list
    def score(self, component_scores):
        return max((component_scores.get(i, 0.0) for i in self.ids), default=0.0)


@dataclass
class K_OF_N(CoverageRule):
    k: int
    ids: list
    def score(self, component_scores):
        vals = sorted((component_scores.get(i, 0.0) for i in self.ids), reverse=True)
        top_k = vals[: self.k]
        return sum(top_k) / self.k if self.k else 1.0


@dataclass
class WEIGHTED(CoverageRule):
    weights: dict   # id -> weight
    def score(self, component_scores):
        total_w = sum(self.weights.values()) or 1.0
        return sum(self.weights[i] * component_scores.get(i, 0.0) for i in self.weights) / total_w


# ---------------------------------------------------------------------------
# Structural constraints
# Each evaluates against a dict: component_id -> Evidence (best match chosen)
# ---------------------------------------------------------------------------

class StructuralConstraint:
    def score(self, evmap: dict, ctx: dict = None) -> float:
        raise NotImplementedError


@dataclass
class NESTED(StructuralConstraint):
    outer: str
    inner: str
    def score(self, evmap, ctx=None):
        outer_ev, inner_ev = evmap.get(self.outer), evmap.get(self.inner)
        if not outer_ev or not inner_ev:
            return 0.0
        # Reject only when outer and inner are literally the SAME Evidence
        # object (the exact same match reused for both slots -- e.g. two
        # slots both resolving to the one cached instance of a lifting
        # wrapper like LoopStep). Comparing whole-object identity, not just
        # .node, is what correctly handles composites: two DIFFERENT
        # composites both have node=None but are still different objects
        # (must be allowed), while the SAME composite resolved once and
        # reused for two slots is the same object (must be rejected). For
        # primitives this is equivalent to the old node-based check, since
        # each detector creates exactly one Evidence object per AST node.
        if outer_ev.start_line <= inner_ev.start_line and inner_ev.end_line <= outer_ev.end_line \
                and outer_ev is not inner_ev:
            return 1.0
        return 0.0


@dataclass
class SEQUENTIAL(StructuralConstraint):
    ids: list
    max_line_gap: int = 3
    def score(self, evmap, ctx=None):
        evs = [evmap.get(i) for i in self.ids]
        if any(e is None for e in evs):
            return 0.0
        evs_sorted = sorted(evs, key=lambda e: e.start_line)
        for a, b in zip(evs_sorted, evs_sorted[1:]):
            if b.start_line - a.end_line > self.max_line_gap:
                return 0.0
        return 1.0


@dataclass
class SHARED_BINDING(StructuralConstraint):
    # list of (id_a, role_a, id_b, role_b); order-independent set match optional
    pairs: list
    as_set: bool = False
    def score(self, evmap, ctx=None):
        matches = 0
        for (ida, ra, idb, rb) in self.pairs:
            ea, eb = evmap.get(ida), evmap.get(idb)
            if not ea or not eb:
                continue
            va, vb = ea.bindings.get(ra), eb.bindings.get(rb)
            if va is None or vb is None:
                continue
            # Plain equality is the original, still-default behavior. When
            # one side is a collection (e.g. TEST's all_names -- every
            # identifier referenced anywhere in a test expression, however
            # it's shaped), treat it as containment instead of exact
            # equality. Purely additive: every pre-existing usage compares
            # two plain strings, where neither branch below ever applies,
            # so nothing already relying on strict equality changes.
            if va == vb:
                matches += 1
            elif isinstance(vb, (set, frozenset, list, tuple)) and va in vb:
                matches += 1
            elif isinstance(va, (set, frozenset, list, tuple)) and vb in va:
                matches += 1
        return matches / len(self.pairs) if self.pairs else 1.0


@dataclass
class SELF_REFERENTIAL(StructuralConstraint):
    """Checks that the ASSIGN's own target variable also appears in its source expr."""
    id: str
    def score(self, evmap, ctx=None):
        ev = evmap.get(self.id)
        if not ev:
            return 0.0
        target = ev.bindings.get("target")
        source = ev.bindings.get("source", "")
        if target and target in source:
            return 1.0
        return 0.0


def _split_tuple_text(s: str) -> list:
    s = s.strip()
    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1]
    return [p.strip() for p in s.split(",") if p.strip()]


@dataclass
class TUPLE_SWAP(StructuralConstraint):
    """Checks that a single ASSIGN evidence is Python's idiomatic tuple swap:
    `a, b = b, a` (one Assign node, target and value both 2-tuples, reversed)."""
    id: str
    def score(self, evmap, ctx=None):
        ev = evmap.get(self.id)
        if not ev:
            return 0.0
        target_parts = _split_tuple_text(ev.bindings.get("target", ""))
        source_parts = _split_tuple_text(ev.bindings.get("source", ""))
        if len(target_parts) == 2 and len(source_parts) == 2 \
                and target_parts == list(reversed(source_parts)):
            return 1.0
        return 0.0


def _resolve_evidence(evmap: dict, dotted_id: str):
    """Resolve 'acc.assign' -> evmap['acc'].children['assign'], so
    constraints can reach into a composite's inner evidence."""
    parts = dotted_id.split(".")
    ev = evmap.get(parts[0])
    for p in parts[1:]:
        if ev is None:
            return None
        ev = ev.children.get(p)
    return ev


@dataclass
class REFERENCES(StructuralConstraint):
    """Checks that a value bound at (source_id, source_role) appears
    textually inside the expression bound at (target_id, target_role).
    E.g. the accumulator variable from Accumulation's inner ASSIGN
    appearing in the OUTPUT expression that's supposed to average it."""
    source_id: str
    source_role: str
    target_id: str
    target_role: str
    def score(self, evmap, ctx=None):
        se = _resolve_evidence(evmap, self.source_id)
        te = _resolve_evidence(evmap, self.target_id)
        if not se or not te:
            return 0.0
        needle = se.bindings.get(self.source_role)
        haystack = te.bindings.get(self.target_role, "") or ""
        return 1.0 if needle and needle in haystack else 0.0


@dataclass
class CONTAINS_TEXT(StructuralConstraint):
    """Checks that a binding's text contains at least one of the given
    substrings -- a deliberately blunt tool (e.g. '/' to confirm an
    OUTPUT expression actually divides, not just prints the running sum)."""
    id: str
    role: str
    substrings: tuple
    def score(self, evmap, ctx=None):
        ev = _resolve_evidence(evmap, self.id)
        if not ev:
            return 0.0
        text = ev.bindings.get(self.role, "") or ""
        return 1.0 if any(s in text for s in self.substrings) else 0.0


def _walk_elif_chain(if_node):
    """Walk forward through an ast.If's elif links (Python represents
    `elif` as a single-element orelse containing another If). Returns
    (branch_count, has_final_else, variable_sets) where variable_sets is
    one frozenset of referenced variable names per branch -- used to check
    whether branches are 'the same decision' even when their tests have
    different shapes (a plain Compare vs. a compound and/or expression),
    rather than requiring an identical single operand string. This is a
    property of ONE AST node's own recursive shape, not a composition of
    separate pieces, which is precisely why it's computed directly rather
    than via nested Advanced competencies (see module docstring note on
    multi-way branching)."""
    def _vars_of(test_expr):
        return frozenset(n.id for n in ast.walk(test_expr) if isinstance(n, ast.Name))

    count = 1
    variable_sets = [_vars_of(if_node.test)]
    node = if_node
    has_else = False
    while True:
        if len(node.orelse) == 1 and isinstance(node.orelse[0], ast.If):
            node = node.orelse[0]
            count += 1
            variable_sets.append(_vars_of(node.test))
        elif len(node.orelse) > 0:
            has_else = True
            break
        else:
            break
    return count, has_else, variable_sets


@dataclass
class EXHAUSTIVE_CHAIN(StructuralConstraint):
    """Checks a TEST evidence's own if/elif/.../else chain: at least
    min_branches total distinguishable outcomes (each If-test, PLUS a
    terminal else if present -- an if/elif/else has 3 outcomes from only
    2 If nodes, which is easy to undercount if you only count If nodes),
    optionally requiring a final else and/or that every branch shares at
    least one common variable across its condition (mutual exclusivity on
    the same decision, not just unrelated ifs that happen to be
    elif-chained) -- checked via shared variables rather than an identical
    operand string, so compound boolean branches (e.g. an `and`/`or` of
    several comparisons) still count as 'the same decision' as long as
    they reference a common variable."""
    id: str
    min_branches: int = 2
    require_else: bool = False
    require_same_operand: bool = False
    def score(self, evmap, ctx=None):
        ev = evmap.get(self.id)
        if not ev or not isinstance(ev.node, ast.If):
            return 0.0
        count, has_else, variable_sets = _walk_elif_chain(ev.node)
        total_outcomes = count + (1 if has_else else 0)
        if total_outcomes < self.min_branches:
            return 0.0
        if self.require_else and not has_else:
            return 0.0
        if self.require_same_operand:
            common = set.intersection(*[set(v) for v in variable_sets]) if variable_sets else set()
            if not common:
                return 0.0
        return 1.0


@dataclass
class AND(StructuralConstraint):
    parts: list
    def score(self, evmap, ctx=None):
        vals = [p.score(evmap, ctx) for p in self.parts]
        return min(vals) if vals else 1.0


@dataclass
class OR(StructuralConstraint):
    parts: list
    def score(self, evmap, ctx=None):
        vals = [p.score(evmap, ctx) for p in self.parts]
        return max(vals) if vals else 0.0


@dataclass
class COOCCUR(StructuralConstraint):
    ids: list
    def score(self, evmap, ctx=None):
        return 1.0 if all(evmap.get(i) for i in self.ids) else 0.0


@dataclass
class DISTINCT_BINDINGS(StructuralConstraint):
    """Checks that no two slots in `ids` share the same combined value across
    `roles` -- e.g. two TEST branches shouldn't have identical
    (operand, op, comparator), or the 'multi-way' branching is really just
    the same condition repeated. Comparing operand alone would misfire on
    the extremely common `if x==1: elif x==2:` (same operand, different
    comparator) -- comparing the full tuple avoids that false negative."""
    ids: list
    roles: tuple
    def score(self, evmap, ctx=None):
        keys = []
        for i in self.ids:
            ev = evmap.get(i)
            if ev is None:
                return 0.0
            keys.append(tuple(ev.bindings.get(r) for r in self.roles))
        return 1.0 if len(set(keys)) == len(keys) else 0.0


@dataclass
class HAS_ELSE(StructuralConstraint):
    """Checks that a TEST evidence's If node ends the chain with a genuine
    terminal `else:` -- not another `elif` (which is itself a nested If in
    orelse) and not simply absent."""
    id: str
    def score(self, evmap, ctx=None):
        ev = evmap.get(self.id)
        if not ev or not isinstance(ev.node, ast.If):
            return 0.0
        orelse = ev.node.orelse
        if not orelse:
            return 0.0
        if len(orelse) == 1 and isinstance(orelse[0], ast.If):
            return 0.0   # that's an elif, not a terminal else
        return 1.0


@dataclass
class DELEGATES(StructuralConstraint):
    """Cross-function composition: the evidence in call_slot must be a CALL
    to a sibling function whose OWN body, independently resolved, satisfies
    callee_competency_id. This is fundamentally different from NESTED/etc,
    which reason about AST containment within one block -- this reasons
    about a call edge between two separately-resolved function scopes.

    Requires ctx = {"model": CompetencyModel, "function_index": {name: FunctionDef}}
    supplied by CompetencyModel.resolve(); falls back to 0.0 without it.
    Memoizes per (callee, competency_id) and guards against recursive calls
    (a function delegating to itself, directly or via a cycle) by treating
    an in-progress resolution as a non-match rather than looping forever.
    """
    call_slot: str
    callee_competency_id: str
    def score(self, evmap, ctx=None):
        ev = evmap.get(self.call_slot)
        if not ev or ctx is None:
            return 0.0
        callee_name = ev.bindings.get("callee")
        func_node = ctx["function_index"].get(callee_name)
        if func_node is None:
            return 0.0
        key = (callee_name, self.callee_competency_id)
        if key in ctx["in_progress"]:
            return 0.0
        if key in ctx["delegate_cache"]:
            return ctx["delegate_cache"][key]
        ctx["in_progress"].add(key)
        try:
            result = ctx["model"].resolve(func_node, [self.callee_competency_id], module_tree=ctx.get("module_tree"))
            score = result[self.callee_competency_id].final_score if self.callee_competency_id in result else 0.0
        finally:
            ctx["in_progress"].discard(key)
        ctx["delegate_cache"][key] = score
        return score


# ---------------------------------------------------------------------------
# Competency definition + resolver
# ---------------------------------------------------------------------------

# DESIGN PRINCIPLE (documented convention, NOT enforced by the code below):
#
# Define a competency's "level" as 0 for a primitive, and
# 1 + max(level(child) for child in components) for a registered
# Competency. When writing NEW competencies, prefer keeping every
# sibling in `components` at the SAME level -- i.e. a level-n competency
# (n > 1) is composed only of level-(n-1) competencies, never a direct
# mix of primitives and higher-level competencies as siblings.
#
# Why this is worth doing when convenient: it gives every competency an
# unambiguous, comparable "depth," useful for sequencing what to
# teach/assess first, and it emerged naturally in most of this library
# without any effort -- Swap, FindExtremum, GreedyChange, and
# BalancedCheck are all clean level-2 competencies purely because of how
# they were written, not because anyone enforced it.
#
# Why it is NOT enforced here: checked directly against the full
# registered library (test_real_data.py + pw01_competencies.py +
# ex2_3_4_competencies.py), 4 of 25 competencies violate it --
# TopKPattern, BubbleSortPass, CountedSearch, and Average -- each because
# one specific piece of raw syntax (a removal step, an output division, a
# search loop) genuinely needed to sit ALONGSIDE a higher-level
# competency as a sibling, not underneath it as a further composition.
# Forcing strict compliance would require trivial "lifting" wrapper
# competencies with zero semantic content, purely to match levels.
#
# A real technical cost if this is ever enforced: a composite Evidence's
# `bindings` dict is always {} (see composite_ev below). NESTED only
# compares line spans, so it's unaffected by wrapping -- but
# SHARED_BINDING, SELF_REFERENTIAL, and TUPLE_SWAP all read
# evmap.get(id).bindings.get(role) directly, and would silently see an
# empty dict once a primitive sits behind lifting wrappers instead of
# being referenced directly. CONTAINS_TEXT and REFERENCES already avoid
# this via dotted-path resolution (_resolve_evidence); the other three
# constraints would need the same treatment before strict layering could
# be safely enforced rather than just documented.

@dataclass
class Competency:
    id: str
    # slot_name -> type_id, where type_id is either:
    #   - a single competency id (primitive or advanced), or
    #   - a tuple of alternative ids (any one of them may fill the slot,
    #     e.g. ("INPUT", "PARAM") for "a value enters here, console or arg")
    # Slot names are what CoverageRule / StructuralConstraint refer to.
    # Using slot names (not raw type ids) is what lets a competency like
    # Swap require three ASSIGN evidences in three distinct roles.
    components: dict
    coverage: CoverageRule = None
    structural: StructuralConstraint = None
    is_primitive: bool = False
    # Strict layering (documented in the level(c) design principle above)
    # says a level-n competency (n > 1) should be composed only of
    # level-(n-1) siblings, never a direct mix of primitives and
    # higher-level competencies. This is enforced at registration time
    # (see CompetencyModel.register) UNLESS this flag is set -- which
    # turns a documented-in-prose exception into a documented-in-code one.
    # A new competency that trips this check without setting the flag is
    # treated as a genuine authoring mistake, not a style preference.
    allow_mixed_levels: bool = False


def _alt_ids(type_id):
    """Normalize a slot's type_id (str or tuple) to a tuple of alternatives."""
    return type_id if isinstance(type_id, tuple) else (type_id,)


class LayeringViolation(ValueError):
    """Raised by CompetencyModel.register() when a new competency mixes
    primitives and higher-level competencies as direct siblings without
    explicitly setting allow_mixed_levels=True."""
    pass


class CompetencyModel:
    def __init__(self):
        self.definitions: dict = {}   # id -> Competency
        self._level_cache: dict = {}

class CompetencyModel:
    def __init__(self):
        self.definitions: dict = {}   # id -> Competency
        self._level_cache: dict = {}

    def _child_levels_list(self, comp: Competency) -> list:
        return [self.level(alt) for t in comp.components.values() for alt in _alt_ids(t)]

    def level(self, cid: str) -> int:
        """0 for a primitive (any id not in self.definitions), otherwise
        1 + max(level of each component). Memoized; register() clears the
        cache since re-registering an id can change its own or others'
        levels."""
        if cid in self._level_cache:
            return self._level_cache[cid]
        if cid not in self.definitions:
            return 0
        comp = self.definitions[cid]
        child_levels = self._child_levels_list(comp)
        lvl = 1 + max(child_levels) if child_levels else 1
        self._level_cache[cid] = lvl
        return lvl

    def child_levels(self, cid: str) -> set:
        return set(self._child_levels_list(self.definitions[cid]))

    def check_layering(self, cid: str = None) -> list:
        """Returns a list of (id, level, child_levels) for every
        registered competency that mixes levels among its direct
        components -- restricted to one id if given, otherwise the whole
        library. Does not consider allow_mixed_levels; use this to audit
        what's currently registered. register() is what actually enforces
        the rule on NEW registrations."""
        ids = [cid] if cid is not None else list(self.definitions.keys())
        violations = []
        for c in ids:
            if c not in self.definitions:
                continue
            cl = self.child_levels(c)
            if len(cl) > 1:
                violations.append((c, self.level(c), sorted(cl)))
        return violations

    def register(self, comp: Competency):
        self._level_cache.clear()
        # Compute directly from the Competency object being registered,
        # NOT via self.definitions[comp.id] -- comp isn't stored yet, so a
        # rejected registration leaves the model genuinely unmodified
        # rather than storing the violator before raising.
        cl = set(self._child_levels_list(comp))
        if len(cl) > 1 and not comp.allow_mixed_levels:
            raise LayeringViolation(
                f"{comp.id} mixes component levels {sorted(cl)}. "
                f"If this is intentional (a primitive genuinely needs to sit alongside a higher-level "
                f"sibling), set allow_mixed_levels=True on the Competency to document that choice "
                f"explicitly; otherwise restructure via a lifting wrapper (see the level(c) design "
                f"principle above)."
            )
        self.definitions[comp.id] = comp

    def _topo_order(self, root_ids):
        order = []
        visited = set()
        temp = set()

        def visit(cid):
            if cid in visited:
                return
            if cid in temp:
                raise ValueError(f"Cycle detected at {cid}")
            if cid not in self.definitions:
                return  # primitive with no separate def entry
            temp.add(cid)
            for type_id in self.definitions[cid].components.values():
                for alt in _alt_ids(type_id):
                    visit(alt)
            temp.discard(cid)
            visited.add(cid)
            order.append(cid)

        for rid in root_ids:
            visit(rid)
        return order

    def resolve_by_scope(self, tree: ast.AST, target_ids: list) -> dict:
        """Resolve target_ids independently per function (plus a synthetic
        '<module>' scope for top-level code outside any function), instead
        of pooling every primitive across the whole file into one flat
        evidence pool. This is what actually prevents SHARED_BINDING (or
        any future variable-name-based check) from matching two unrelated
        functions that happen to use the same variable name -- resolve()
        alone only protects nested-containment checks like NESTED, which
        naturally can't span two functions' non-overlapping line ranges,
        but SHARED_BINDING has no such protection on its own.

        Returns {scope_name: {competency_id: Result}}. Cross-function
        DELEGATES still works inside each per-function resolve() call,
        because the sibling function index is always built from the whole
        module_tree regardless of which function's primitives are in scope.
        """
        if not isinstance(tree, ast.Module):
            # Nothing to partition -- already a single function/snippet.
            return {"<single>": self.resolve(tree, target_ids, module_tree=tree)}

        results = {}
        top_level_functions = [n for n in tree.body if isinstance(n, ast.FunctionDef)]
        for func in top_level_functions:
            results[func.name] = self.resolve(func, target_ids, module_tree=tree)

        non_function_stmts = [n for n in tree.body if not isinstance(n, ast.FunctionDef)]
        if non_function_stmts:
            module_scope = ast.Module(body=non_function_stmts, type_ignores=[])
            results["<module>"] = self.resolve(module_scope, target_ids, module_tree=tree)

        return results

    def resolve(self, tree: ast.AST, target_ids: list, module_tree: ast.AST = None) -> dict:
        import itertools
        pidx = _ParentIndex(tree)

        # 1. run primitive detectors once, scoped to `tree` -- when called
        # via resolve_by_scope, `tree` is a single function's subtree, so
        # DECL/LOOP/TEST/ASSIGN/etc. evidence can never spuriously match
        # across two unrelated functions (SHARED_BINDING in particular has
        # no location awareness of its own; scoping the detector pass is
        # what actually prevents cross-function name collisions, not NESTED,
        # which only protects nested-containment checks).
        primitive_evidence = {pid: fn(tree, pidx) for pid, fn in PRIMITIVE_DETECTORS.items()}

        # 1b. Sibling function lookup for CALL/DELEGATES is intentionally
        # NOT scoped to `tree` -- it always comes from module_tree (the
        # whole file), defaulting to `tree` itself when not given (e.g. the
        # original single-call usage, or resolving a function in isolation
        # with no siblings in view). This is what lets per-function primitive
        # scoping and cross-function delegation coexist: a function can be
        # analyzed in isolation for its own DECL/LOOP/etc., while still being
        # able to delegate to a sibling defined elsewhere in the same file.
        module_tree = module_tree if module_tree is not None else tree
        function_index = {n.name: n for n in ast.walk(module_tree) if isinstance(n, ast.FunctionDef)}
        call_evidence = []
        for node in ast.walk(tree):
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) \
                    and node.func.id in function_index:
                s, e = _line_span(node)
                call_evidence.append(Evidence("CALL", node, s, e,
                                               bindings={"callee": node.func.id},
                                               enclosing=pidx.enclosing_block(node)))
        primitive_evidence["CALL"] = call_evidence

        # Tag every piece of evidence with its enclosing function (None =
        # module-level). This is what lets the resolver refuse to combine
        # evidence from two unrelated functions just because a structural
        # constraint (typically SHARED_BINDING, which has no line-based
        # containment check of its own) happens to match on variable name.
        for evs in primitive_evidence.values():
            for ev in evs:
                if ev.node is None:
                    continue
                if ev.competency_id == "PARAM" and isinstance(ev.node, ast.FunctionDef):
                    # PARAM stores the FunctionDef itself as .node (there's
                    # no separate AST node per-argument worth tracking), so
                    # enclosing_function would walk PAST it to the parent
                    # scope -- a parameter belongs to the function it's a
                    # parameter of, not that function's enclosing context.
                    ev.scope = ev.node
                else:
                    ev.scope = pidx.enclosing_function(ev.node)

        ctx = {"model": self, "function_index": function_index, "module_tree": module_tree,
               "delegate_cache": {}, "in_progress": set()}

        cache: dict = {}   # competency_id -> Result

        order = self._topo_order(target_ids)
        for cid in order:
            comp = self.definitions[cid]

            # Split slots into: fixed (resolves to a single advanced-competency
            # evidence) vs. candidate lists (one or more primitive occurrences,
            # possibly pooled across alternative types for an OR-slot).
            fixed = {}
            candidate_lists = {}
            for slot, type_id in comp.components.items():
                alts = _alt_ids(type_id)
                if len(alts) == 1 and alts[0] in cache:
                    res = cache[alts[0]]
                    has_evidence = res.final_score > 0 and res.evidences
                    fixed[slot] = (res.final_score, res.evidences[0] if has_evidence else None)
                else:
                    pool = []
                    for alt in alts:
                        pool.extend(primitive_evidence.get(alt, []))
                    candidate_lists[slot] = pool if pool else [None]

            # Search combinations of primitive evidence (capped to keep it cheap)
            keys = list(candidate_lists.keys())
            combos = list(itertools.product(*[candidate_lists[k] for k in keys])) if keys else [()]
            MAX_COMBOS = 300
            if len(combos) > MAX_COMBOS:
                combos = combos[:MAX_COMBOS]

            best = None  # (final_score, coverage_score, structural_score, evmap)
            fixed_scopes = {ev.scope for (_, ev) in fixed.values() if ev is not None and ev.scope is not None}
            for combo in combos:
                # skip combos that reuse the exact same AST node in two slots
                nodes_used = [ev.node for ev in combo if ev is not None]
                if len(nodes_used) != len(set(id(n) for n in nodes_used)):
                    continue

                # skip combos that mix evidence from two different functions
                # (a coincidental variable-name match across unrelated
                # functions should not satisfy a competency)
                combo_scopes = {ev.scope for ev in combo if ev is not None and ev.scope is not None}
                if len(fixed_scopes | combo_scopes) > 1:
                    continue

                evmap = {slot: ev for slot, (_, ev) in fixed.items() if ev is not None}
                component_scores = {slot: score for slot, (score, _) in fixed.items()}
                for slot, ev in zip(keys, combo):
                    component_scores[slot] = 1.0 if ev is not None else 0.0
                    if ev is not None:
                        evmap[slot] = ev

                coverage_score = comp.coverage.score(component_scores) if comp.coverage else 1.0
                structural_score = comp.structural.score(evmap, ctx) if comp.structural else 1.0
                final_score = coverage_score * structural_score
                if best is None or final_score > best[0]:
                    best = (final_score, coverage_score, structural_score, evmap)

            if best is None:
                # every combo was rejected by the node-reuse guard; fall back
                # to the empty combination so we still report 0 rather than crash
                evmap = {slot: ev for slot, (_, ev) in fixed.items() if ev is not None}
                component_scores = {slot: score for slot, (score, _) in fixed.items()}
                for slot in keys:
                    component_scores.setdefault(slot, 0.0)
                coverage_score = comp.coverage.score(component_scores) if comp.coverage else 1.0
                structural_score = comp.structural.score(evmap, ctx) if comp.structural else 1.0
                best = (coverage_score * structural_score, coverage_score, structural_score, evmap)

            final_score, coverage_score, structural_score, evmap = best

            if evmap:
                lines = [e.start_line for e in evmap.values()] + [e.end_line for e in evmap.values()]
                span = (min(lines), max(lines))
            else:
                span = (0, 0)

            composite_ev = Evidence(cid, None, span[0], span[1],
                                     bindings={}, score=final_score, children=evmap)
            child_scopes = {e.scope for e in evmap.values() if e.scope is not None}
            composite_ev.scope = child_scopes.pop() if len(child_scopes) == 1 else None

            detail = f"coverage={coverage_score:.2f} structural={structural_score:.2f}"
            cache[cid] = Result(cid, coverage_score, structural_score, final_score,
                                 [composite_ev], detail)

        return {tid: cache[tid] for tid in target_ids if tid in cache}
'''

with open('competency.py', 'w') as f:
    f.write(competency_source)

test_real_data_source = r'''
import ast
import glob
from competency import (
    CompetencyModel, Competency, ALL, ANY, NESTED, SHARED_BINDING,
    SELF_REFERENTIAL, TUPLE_SWAP, CONTAINS_TEXT, AND, SEQUENTIAL, DELEGATES
)

model = CompetencyModel()

model.register(Competency(
    id="Accumulation",
    components={"decl": "DECL", "loop": "LOOP", "assign": "ASSIGN"},
    coverage=ALL(["decl", "loop", "assign"]),
    structural=AND([
        NESTED("loop", "assign"),
        SHARED_BINDING([("decl", "target", "assign", "target")]),
        SELF_REFERENTIAL("assign"),
    ]),
))

# Swap now recognizes two distinct correct forms: the 3-statement temp-variable
# version, and Python's idiomatic tuple-unpack `a, b = b, a` (a single ASSIGN
# node) -- real question_4 submissions use both, and the tuple form was
# invisible to the model until now.
model.register(Competency(
    id="Swap3Step",
    components={"tmp": "ASSIGN", "first": "ASSIGN", "second": "ASSIGN"},
    coverage=ALL(["tmp", "first", "second"]),
    structural=AND([
        SEQUENTIAL(["tmp", "first", "second"], max_line_gap=1),
        SHARED_BINDING([
            ("tmp", "source", "first", "target"),
            ("first", "source", "second", "target"),
            ("second", "source", "tmp", "target"),
        ]),
    ]),
))

model.register(Competency(
    id="SwapTuple",
    components={"assign": "ASSIGN"},
    coverage=ALL(["assign"]),
    structural=TUPLE_SWAP("assign"),
))

model.register(Competency(
    id="Swap",
    components={"three_step": "Swap3Step", "tuple_form": "SwapTuple"},
    coverage=ANY(["three_step", "tuple_form"]),
    structural=None,
))

# "source" slot now accepts either INPUT (console) or PARAM (function arg) --
# fixes the false-zero on every function-graded submission from last run.
#
# SHARED_BINDING now checks against TEST's all_names (every identifier
# referenced anywhere in the test, however it's shaped) rather than just
# operand (the literal left-hand side of a simple comparison). Batch
# validation against the real Question 1 dataset found this matters a lot:
# 364 of 526 zero-scoring submissions had every slot present but still
# scored 0, because the searched value appeared on the right side of a
# comparison ("elem >= x", 55% of cases), inside a compound "and"/"or" or
# chained condition ("x > seq[i] and x <= seq[i+1]", 43%), or as a while
# loop's own condition rather than a nested if (2%) -- none of which the
# old operand-only check could see. all_names is a strict superset of what
# operand alone matched, so nothing that used to pass stops passing.
model.register(Competency(
    id="GuardedInput",
    components={"source": ("INPUT", "PARAM"), "test": "TEST", "loop": "LOOP"},
    coverage=ALL(["source", "test", "loop"]),
    structural=AND([
        NESTED("loop", "test"),
        SHARED_BINDING([("source", "target", "test", "all_names")]),
    ]),
))

model.register(Competency(
    id="SelectionStep",
    components={"loop": "LOOP", "test": "TEST", "assign": "ASSIGN"},
    coverage=ALL(["loop", "test", "assign"]),
    structural=AND([
        NESTED("loop", "test"),
        NESTED("test", "assign"),
    ]),
))

# Whether "found the extremum via max()/min()" should count the same as
# "found it via a manual comparison loop" is a modeling choice -- ~30% of
# real Top-K submissions use the builtin. Rather than picking one answer,
# expose both forms and let the composite ANY over them; the evidence
# (which branch actually matched) is still inspectable per-submission.
model.register(Competency(
    id="FindExtremumBuiltin",
    components={"call": "BUILTIN_REDUCE"},
    coverage=ALL(["call"]),
    structural=None,
))

model.register(Competency(
    id="FindExtremum",
    components={"manual": "SelectionStep", "builtin": "FindExtremumBuiltin"},
    coverage=ANY(["manual", "builtin"]),
    structural=None,
))

model.register(Competency(
    id="LoopStep", components={"inner": "LOOP"}, coverage=ALL(["inner"]), structural=None,
))
model.register(Competency(
    id="LoopStep2", components={"inner": "LoopStep"}, coverage=ALL(["inner"]), structural=None,
))
model.register(Competency(
    id="RemovalAssign",
    components={"assign": "ASSIGN"},
    coverage=ALL(["assign"]),
    # the ".remove()/.pop() specifically" check now lives HERE, one level
    # down, instead of via a dotted path from TopKPattern itself.
    structural=CONTAINS_TEXT("assign", "kind", ("remove", "pop")),
))
model.register(Competency(
    id="RemovalAssignStep", components={"inner": "RemovalAssign"}, coverage=ALL(["inner"]), structural=None,
))
model.register(Competency(
    id="TopKPattern",
    components={"loop": "LoopStep2", "select": "FindExtremum", "remove": "RemovalAssignStep"},
    coverage=ALL(["loop", "select", "remove"]),
    structural=AND([
        NESTED("loop", "select"),
        NESTED("loop", "remove"),
    ]),
))

# Real-data finding: clustering the "novel" (score-0) Q5 submissions by
# embedding similarity found one dominant cluster (38 of 43 files) sharing
# a shape TopKPattern, defined purely inline, cannot see at all: top_k's
# OWN body does nothing but call a differently-named helper and slice the
# result --
#     def top_k(lst, k):
#         return sort_descending(lst)[:k]
# -- so function-scoped resolution correctly finds zero loop/test/assign
# inside top_k itself. Two genuinely different sub-shapes turned up inside
# that one cluster:
#
# (1) The helper itself repeatedly finds-and-removes the max -- structurally
#     IDENTICAL to TopKPattern's own existing definition, just living in a
#     function with a different name. DELEGATES (already built for exactly
#     this) recognizes it directly, checking the callee's own body against
#     TopKPattern with zero new detection logic needed.
model.register(Competency(
    id="TopKViaDelegate",
    components={"call": "CALL"},
    coverage=ALL(["call"]),
    structural=DELEGATES("call", "TopKPattern"),
))

# (2) The helper is a genuinely different, general-purpose sort (recursive
#     merge_sort, an in-place bubble-style sort) -- not a repeated-extraction
#     shape at all, so DELEGATES-to-TopKPattern would correctly score this
#     0. "Sort everything, then take the first k" is its own legitimate
#     strategy. HasSubstantialWork checks for a loop OR self-recursion --
#     rules out delegating to a stub, without trying to characterize "is
#     this a valid general sort," a much bigger question than this
#     competency needs to answer. Recursion matters here specifically: a
#     real submission's merge_sort was found to have ZERO LOOP primitives
#     at all (pure recursion), so a loop-only check misses genuinely
#     substantial recursive work entirely.
model.register(Competency(
    id="HasSubstantialWork",
    components={"loop": ("LOOP", "SELF_RECURSIVE")},
    coverage=ALL(["loop"]),
    structural=None,
))
model.register(Competency(
    id="SortThenSlice",
    components={"call": "CALL", "output": "OUTPUT"},
    coverage=ALL(["call", "output"]),
    structural=AND([
        DELEGATES("call", "HasSubstantialWork"),
        # Generous gap: real submissions in this shape sometimes have a
        # couple of intermediate statements between the call and the
        # final return (e.g. a .reverse() call, or building the sliced
        # result via a small loop rather than a literal [:k]), not just
        # a single one-line return.
        SEQUENTIAL(["call", "output"], max_line_gap=10),
    ]),
))

# Master competency: the original inline shape, plus both delegate shapes,
# as an explicit ANY -- same pattern as SortByKey. Registered separately
# from TopKPattern itself (rather than redefining TopKPattern in place) so
# existing code/results referencing "TopKPattern" specifically are
# unaffected; batch validation should target TopKPatternExtended to see
# the corrected coverage.
#
# Documented layering exception, same convention as BubbleSortPass: the two
# delegate-based branches are inherently shallower than the inline
# TopKPattern branch, because their real depth lives inside DELEGATES' own
# recursive resolution of the callee's body -- which correctly does not
# propagate into level(), since it is a separate, independently-scoped
# resolution rather than a component of THIS competency's own structure.
model.register(Competency(
    id="TopKPatternExtended",
    components={"direct": "TopKPattern", "delegate": "TopKViaDelegate", "sort_then_slice": "SortThenSlice"},
    coverage=ANY(["direct", "delegate", "sort_then_slice"]),
    structural=None,
    allow_mixed_levels=True,
))

# Question 2 pattern: count occurrences matching a condition. Note this is
# defined directly over primitives rather than composing Accumulation inside
# a TEST: Accumulation's own span includes its DECL line, which legitimately
# sits BEFORE/outside the TEST (only the increment belongs inside it), so a
# naive NESTED(test, Accumulation) on full composite span breaks. Composing
# advanced-in-advanced only works cleanly when the child's whole span is
# meant to be contained -- that wasn't true here.
model.register(Competency(
    id="ConditionalCount",
    components={"decl": "DECL", "loop": "LOOP", "test": "TEST", "increment": "ASSIGN"},
    coverage=ALL(["decl", "loop", "test", "increment"]),
    structural=AND([
        NESTED("loop", "test"),
        NESTED("test", "increment"),
        SHARED_BINDING([("decl", "target", "increment", "target")]),
        SELF_REFERENTIAL("increment"),
    ]),
))

# Real bubble sort: outer LOOP { inner LOOP { TEST { Swap } } } -- Swap nested
# *inside* the comparison, immediately swapping on the adjacent pair. This is
# structurally distinct from "selection sort" (find extremum across the whole
# inner loop, swap once *after* it ends) even though both use two LOOPs,
# a TEST, and a Swap.
model.register(Competency(
    id="BubbleSortPass",
    components={"outer": "LOOP", "inner": "LOOP", "test": "TEST", "swap": "Swap"},
    coverage=ALL(["outer", "inner", "test", "swap"]),
    structural=AND([
        NESTED("outer", "inner"),
        NESTED("inner", "test"),
        NESTED("test", "swap"),
    ]),
    # Documented layering exception: Swap (level 2) sits directly alongside
    # the primitives that locate it, matching the actual AST shape of a
    # bubble-sort pass -- see the level(c) design principle in competency.py.
    allow_mixed_levels=True,
))

# SortByKey: the real Q4 target competency. Batch validation against all
# 419 real sort_age submissions found BubbleSortPass (in-place swap, the
# reference solution's own strategy) covers only 13.1% -- most real
# submissions instead repeatedly find the max and remove it, building a
# new sorted list, i.e. exactly the TopKPattern shape already built for
# Q5's top_k (64.4% alone, requiring zero new code). Combined: 77.6%.
model.register(Competency(
    id="SortByKey",
    components={"inplace_swap": "BubbleSortPass", "repeated_extraction": "TopKPattern"},
    coverage=ANY(["inplace_swap", "repeated_extraction"]),
    structural=None,
))


def analyze(path, targets):
    src = open(path, encoding="utf-8").read()
    try:
        tree = ast.parse(src)
    except SyntaxError as e:
        return None, f"SyntaxError: {e}"
    results = model.resolve(tree, targets)
    return results, None


def show(path, targets, src_override=None):
    print(f"\n{'='*70}\n{path}\n{'='*70}")
    if src_override:
        print(src_override)
        results = model.resolve(ast.parse(src_override), targets)
        err = None
    else:
        print(open(path, encoding="utf-8").read())
        results, err = analyze(path, targets)
    if err:
        print(err)
        return
    for tid, r in results.items():
        print(f"  {tid:16s} final={r.final_score:.2f}  ({r.detail})")


if __name__ == "__main__":
    print("\n\n########## Fix check 1: Swap (needs 3 distinct ASSIGN slots) ##########")
    good_swap = """
def f(a, b):
    temp = a
    a = b
    b = temp
    return a, b
"""
    buggy_swap = """
def f(a, b):
    temp = a
    b = a
    a = temp
    return a, b
"""
    show("synthetic: correct swap", ["Swap"], src_override=good_swap)
    show("synthetic: buggy swap (loses original b)", ["Swap"], src_override=buggy_swap)

    print("\n\n########## Fix check 2: GuardedInput on parameter-based code ##########")
    files1 = sorted(glob.glob("/home/claude/refactory/data/question_1/code/correct/*.py"))
    for f in [files1[0], files1[1], files1[2]]:
        show(f, ["GuardedInput"])

    print("\n\n########## Re-check question 5 still works ##########")
    files5 = sorted(glob.glob("/home/claude/refactory/data/question_5/code/correct/*.py"))
    for f in [files5[0], files5[49], files5[299]]:
        show(f, ["SelectionStep", "TopKPattern"])

    print("\n\n########## Fix check 3: BubbleSortPass on real question_4 code ##########")
    q4 = "/home/claude/refactory/data/question_4/code/correct"
    show(f"{q4}/correct_4_121.py", ["Swap", "BubbleSortPass"])   # tuple-swap bubble sort
    show(f"{q4}/correct_4_164.py", ["Swap", "BubbleSortPass"])   # tuple-swap bubble sort
    show(f"{q4}/correct_4_043.py", ["Swap", "BubbleSortPass"])   # selection sort w/ swap AFTER inner loop
    show(f"{q4}/correct_4_010.py", ["Swap", "BubbleSortPass"])   # selection sort, no swap at all

    print("\n\n########## Fix check 4: FindExtremum (manual vs. builtin max/min) ##########")
    q5 = "/home/claude/refactory/data/question_5/code/correct"
    show(f"{q5}/correct_5_001.py", ["FindExtremum", "TopKPattern"])   # builtin max()/remove()
    show(f"{q5}/correct_5_050.py", ["FindExtremum", "TopKPattern"])   # manual comparison loop

    print("\n\n########## Question 2: ConditionalCount (composes Accumulation) ##########")
    files2 = sorted(glob.glob("/home/claude/refactory/data/question_2/code/correct/*.py"))
    for f in [files2[0], files2[1]]:
        show(f, ["ConditionalCount"])

    print("\n\n########## Question 3: SelectionStep reused as-is (remove_extras) ##########")
    files3 = sorted(glob.glob("/home/claude/refactory/data/question_3/code/correct/*.py"))
    for f in [files3[0], files3[1]]:
        show(f, ["SelectionStep"])
'''

with open('test_real_data.py', 'w') as f:
    f.write(test_real_data_source)

pw01_competencies_source = r'''
import ast
from competency import Competency, ALL, NESTED, AND, REFERENCES, CONTAINS_TEXT, DELEGATES
from test_real_data import model, show

# --- Ex1.3/1.4: next N primes -----------------------------------------
# IsPrime: a LOOP that TESTs divisibility (operand must involve '%').
model.register(Competency(
    id="IsPrime",
    components={"loop": "LOOP", "test": "TEST"},
    coverage=ALL(["loop", "test"]),
    structural=AND([
        NESTED("loop", "test"),
        CONTAINS_TEXT("test", "operand", ("%",)),
    ]),
))

# LoopStep / OutputStep: trivial level-1 lifts of LOOP/OUTPUT, registered
# here (before CountedSearch and Average, both of which need them) so
# neither depends on import order from another file.
model.register(Competency(
    id="LoopStep",
    components={"inner": "LOOP"},
    coverage=ALL(["inner"]),
    structural=None,
))

model.register(Competency(
    id="OutputStep",
    components={"out": "OUTPUT"},
    coverage=ALL(["out"]),
    structural=None,
))

# CountedSearch: an outer LOOP that both runs a primality check and
# produces output, on each iteration -- deliberately loose (siblings, not
# nested in each other) since "print inside the search" vs "collect then
# print after" are both reasonable strategies; a first-pass approximation,
# to be tightened once real student submissions show which shapes recur.
# Kept level-2-clean via LoopStep/OutputStep rather than mixing primitives
# directly alongside IsPrime (level 1) as siblings.
model.register(Competency(
    id="CountedSearch",
    components={"outer": "LoopStep", "isprime": "IsPrime", "output": "OutputStep"},
    coverage=ALL(["outer", "isprime", "output"]),
    structural=AND([
        NESTED("outer", "isprime"),
        NESTED("outer", "output"),
    ]),
))

# --- Ex1.5: collect N inputs into a list --------------------------------
model.register(Competency(
    id="CollectInput",
    components={"loop": "LOOP", "input": "INPUT", "assign": "ASSIGN"},
    coverage=ALL(["loop", "input", "assign"]),
    structural=AND([
        NESTED("loop", "input"),
        NESTED("loop", "assign"),
    ]),
))

# --- Ex1.2: average of n notes ------------------------------------------
# Reuses Accumulation (already in the shared library) and adds: the
# accumulator variable must actually flow into a division in the output.
model.register(Competency(
    id="Average",
    components={"acc": "Accumulation", "output": "OutputStep"},
    coverage=ALL(["acc", "output"]),
    structural=AND([
        REFERENCES("acc.assign", "target", "output.out", "source"),
        CONTAINS_TEXT("output.out", "source", ("/",)),
    ]),
))


# --- Question 2 (contains_unique_day): cross-function delegation ---------
# contains_unique_day doesn't count anything itself -- it delegates the
# uniqueness check to unique_day. DELEGATES independently resolves
# ConditionalCount against the CALLED function's own body, so the score
# here reflects whether the callee genuinely implements it, not just
# whether a call exists.
model.register(Competency(
    id="DelegatesUniqueness",
    components={"test": "TEST", "call": "CALL"},
    coverage=ALL(["test", "call"]),
    structural=AND([
        NESTED("test", "call"),
        DELEGATES("call", "ConditionalCount"),
    ]),
))


if __name__ == "__main__":
    print("\n\n########## Ex1.2: average (reference) ##########")
    show_src = """
def moyenne(n):
    total = 0
    for i in range(n):
        note = float(input("Note: "))
        total += note
    return total / n
"""
    print(show_src)
    r = model.resolve(ast.parse(show_src), ["Accumulation", "Average"])
    for tid, res in r.items():
        print(f"  {tid:16s} final={res.final_score:.2f}  ({res.detail})")

    print("\n\n########## Ex1.2: average (buggy -- forgets the division) ##########")
    buggy_src = """
def moyenne(n):
    total = 0
    for i in range(n):
        note = float(input("Note: "))
        total += note
    return total
"""
    print(buggy_src)
    r = model.resolve(ast.parse(buggy_src), ["Accumulation", "Average"])
    for tid, res in r.items():
        print(f"  {tid:16s} final={res.final_score:.2f}  ({res.detail})")

    print("\n\n########## Ex1.3/1.4: next N primes ##########")
    primes_src = """
def nb_premiers(nb, n=3):
    count = 0
    candidate = nb + 1
    while count < n:
        is_prime = True
        for d in range(2, candidate):
            if candidate % d == 0:
                is_prime = False
        if is_prime:
            print(candidate)
            count += 1
        candidate += 1
"""
    print(primes_src)
    r = model.resolve(ast.parse(primes_src), ["IsPrime", "CountedSearch"])
    for tid, res in r.items():
        print(f"  {tid:16s} final={res.final_score:.2f}  ({res.detail})")

    print("\n\n########## Ex1.5: collect module names into a list ##########")
    collect_src = """
def lire_modules():
    A = []
    for i in range(5):
        nom = input("Module: ")
        A.append(nom)
    return A
"""
    print(collect_src)
    r = model.resolve(ast.parse(collect_src), ["CollectInput"])
    for tid, res in r.items():
        print(f"  {tid:16s} final={res.final_score:.2f}  ({res.detail})")

    print("\n\n########## Ex1.6: longest string (manual) -- reuses SelectionStep ##########")
    longest_manual = """
def plus_long(liste):
    longest = liste[0]
    for s in liste:
        if len(s) > len(longest):
            longest = s
    return longest
"""
    print(longest_manual)
    r = model.resolve(ast.parse(longest_manual), ["SelectionStep", "FindExtremum"])
    for tid, res in r.items():
        print(f"  {tid:16s} final={res.final_score:.2f}  ({res.detail})")

    print("\n\n########## Ex1.6: longest string (builtin) -- reuses FindExtremum ##########")
    longest_builtin = """
def plus_long(liste):
    return max(liste, key=len)
"""
    print(longest_builtin)
    r = model.resolve(ast.parse(longest_builtin), ["SelectionStep", "FindExtremum"])
    for tid, res in r.items():
        print(f"  {tid:16s} final={res.final_score:.2f}  ({res.detail})")

    print("\n\n########## Cross-function delegation: contains_unique_day -> unique_day ##########")
    q2_delegation_src = """
def unique_day(day, possible_birthdays):
    count = 0
    for birthday in possible_birthdays:
        if birthday[1] == day:
            count += 1
    return count == 1

def contains_unique_day(month, possible_birthdays):
    for birthday in possible_birthdays:
        if birthday[0] == month:
            if unique_day(birthday[1], possible_birthdays):
                return True
    return False
"""
    print(q2_delegation_src)
    r = model.resolve(ast.parse(q2_delegation_src), ["ConditionalCount", "DelegatesUniqueness"])
    for tid, res in r.items():
        print(f"  {tid:20s} final={res.final_score:.2f}  ({res.detail})")
'''

with open('pw01_competencies.py', 'w') as f:
    f.write(pw01_competencies_source)

q2_competencies_source = r'''
import ast
from competency import Competency, ALL, ANY, AND, REFERENCES, CONTAINS_TEXT
from pw01_competencies import model  # registers ConditionalCount, Accumulation, etc.

# Trivial level-1 lift of OUTPUT, same technique used for AverageStrict earlier --
# keeps AccumulateThenCount at clean level 2 (Accumulation + OutputStep, both
# level 1) rather than mixing a level-1 competency with a raw primitive.
model.register(Competency(
    id="OutputStep",
    components={"out": "OUTPUT"},
    coverage=ALL(["out"]),
    structural=None,
))

# The real alternative strategy found in Q2 data (correct_2_001.py and siblings):
# accumulate matching values into a collection, then check membership count on
# the collection directly, instead of an explicit counter.
model.register(Competency(
    id="AccumulateThenCount",
    components={"acc": "Accumulation", "check": "OutputStep"},
    coverage=ALL(["acc", "check"]),
    structural=AND([
        # the accumulated collection must be the SAME one being counted --
        # not just "some accumulation and some .count() exist somewhere"
        REFERENCES("acc.decl", "target", "check.out", "source"),
        CONTAINS_TEXT("check.out", "source", (".count(",)),
    ]),
))

# Trivial level-2 lift of ConditionalCount (level 1), so both branches of
# the ANY below are the same level (2) -- keeps UniqueCheck cleanly level 3
# instead of adding a third documented layering exception alongside
# TopKPattern and BubbleSortPass.
model.register(Competency(
    id="ConditionalCountStep",
    components={"inner": "ConditionalCount"},
    coverage=ALL(["inner"]),
    structural=None,
))

# --- Third real strategy found in the data: functional map()/filter()
# pipelines with no explicit loop at all. Modeled via CONTAINS_TEXT/
# REFERENCES on the existing ASSIGN/TEST/OUTPUT primitives directly --
# no new primitive detector needed. Three real sub-shapes found:
#   (a) one-liner: return tuple(map(...)).count(day) == 1
#   (b) assign-then-check: possible = tuple(filter(...)); if len(possible) == 1: ...
#       (or the map()+.count() equivalent of the same two-statement shape)
# Both checked via plain substring matching on the relevant expression
# text rather than a dedicated MAP/FILTER primitive -- map()/filter() are
# ordinary Call nodes already visible inside ASSIGN.source / OUTPUT.source,
# so no new primitive detector is needed.

model.register(Competency(
    id="InlineFunctionalCheck",           # shape (a): everything in one return
    components={"output": "OUTPUT"},
    coverage=ALL(["output"]),
    structural=AND([
        CONTAINS_TEXT("output", "source", ("map(", "filter(")),
        CONTAINS_TEXT("output", "source", (".count(", "len(")),
    ]),
))

model.register(Competency(
    id="GuardedFunctionalCheck",          # shape (b): build the collection, then check it
    components={"assign": "ASSIGN", "test": "TEST"},
    coverage=ALL(["assign", "test"]),
    structural=AND([
        CONTAINS_TEXT("assign", "source", ("map(", "filter(")),
        CONTAINS_TEXT("test", "operand", (".count(", "len(")),
        # the value being tested must be the SAME one the pipeline built,
        # not just "a pipeline and a check exist somewhere in the function"
        REFERENCES("assign", "target", "test", "operand"),
    ]),
))

model.register(Competency(
    id="InlineTestFunctionalCheck",       # shape (c): pipeline embedded directly
    components={"test": "TEST"},          # in the if-condition itself, e.g.
    coverage=ALL(["test"]),               # if (tuple(map(...))).count(day) == 1:
    structural=AND([
        CONTAINS_TEXT("test", "operand", ("map(", "filter(")),
        CONTAINS_TEXT("test", "operand", (".count(", "len(")),
    ]),
))

model.register(Competency(
    id="FunctionalUniqueCheck",
    components={"inline": "InlineFunctionalCheck", "guarded": "GuardedFunctionalCheck",
                "inline_test": "InlineTestFunctionalCheck"},
    coverage=ANY(["inline", "guarded", "inline_test"]),
    structural=None,
))

# Master competency, now with all three real strategies as same-level
# (level 2) siblings under one ANY, keeping UniqueCheck cleanly level 3.
model.register(Competency(
    id="UniqueCheck",
    components={"counter": "ConditionalCountStep", "accumulate_count": "AccumulateThenCount",
                "functional": "FunctionalUniqueCheck"},
    coverage=ANY(["counter", "accumulate_count", "functional"]),
    structural=None,
))


if __name__ == "__main__":
    print("########## AccumulateThenCount on the real tuple+count strategy ##########")
    src_tuple = """
def unique_day(day, possible_birthdays):
    all_days = ()
    for i in possible_birthdays:
        all_days += (i[1],)
    return all_days.count(day) == 1
"""
    print(src_tuple)
    r = model.resolve(ast.parse(src_tuple), ["ConditionalCount", "AccumulateThenCount", "UniqueCheck"])
    for cid, res in r.items():
        print(f"  {cid:20s} final={res.final_score:.2f}  ({res.detail})")

    print("\n########## UniqueCheck on the counter strategy (should also pass) ##########")
    src_counter = """
def unique_day(day, possible_birthdays):
    count = 0
    for birthday in possible_birthdays:
        if birthday[1] == day:
            count += 1
    return count == 1
"""
    print(src_counter)
    r2 = model.resolve(ast.parse(src_counter), ["ConditionalCount", "AccumulateThenCount", "UniqueCheck"])
    for cid, res in r2.items():
        print(f"  {cid:20s} final={res.final_score:.2f}  ({res.detail})")
'''

with open('q2_competencies.py', 'w') as f:
    f.write(q2_competencies_source)

report_source = r'''
"""
Per-submission competency report.

Two layers, matching the model itself:
  - Primitive inventory: every primitive construct found, with its
    location and a short human-readable description.
  - Advanced competency scorecard: for a chosen list of target
    competency ids, the resolved coverage/structural/final scores, PLUS
    which named slots were actually satisfied vs. missing -- this part
    is only possible because composite Evidence keeps a `children` map,
    so we can compare a competency's declared slots against which ones
    actually got evidence.

Known limitation, stated plainly rather than glossed over: when a
composite's STRUCTURAL score is < 1.0 but coverage is 1.0 (all pieces
present, wrong arrangement), the report can currently only surface the
aggregate structural score, not which specific sub-constraint inside an
AND/OR failed. Constraints don't yet expose per-part results, only a
combined float. Surfacing that would need each StructuralConstraint to
optionally report its own breakdown, not just a score -- worth doing if
this report is used for actual student-facing feedback, since "structure
failed" alone doesn't tell a student what to fix.
"""

import ast
from competency import PRIMITIVE_DETECTORS, _ParentIndex, CompetencyModel
from semantic_backup import semantic_backup_score


def _describe(ev) -> str:
    """One short human-readable line per evidence, by primitive type."""
    b = ev.bindings
    cid = ev.competency_id
    if cid == "DECL":
        return f"{b.get('target')} declared"
    if cid == "INPUT":
        return f"input() -> {b.get('target')}"
    if cid == "PARAM":
        return f"parameter {b.get('target')}"
    if cid == "FUNC_DEF":
        ret = "returns a value" if b.get("has_return") == "yes" else "no return value"
        return f"def {b.get('name')}({b.get('n_params')} params), {ret}"
    if cid == "OUTPUT":
        return f"output: {b.get('source')}"
    if cid == "TEST":
        if b.get("comparator") is not None:
            return f"if {b.get('operand')} {b.get('op')} {b.get('comparator')}"
        return f"if {b.get('operand')}"
    if cid == "LOOP":
        if b.get("kind") == "for":
            return f"for {b.get('iterator_var')} in {b.get('bound_expr')}"
        return f"while {b.get('bound_expr')}"
    if cid == "ASSIGN":
        kind = b.get("kind")
        if kind in ("append", "add", "pop", "remove"):
            return f"{b.get('target')}.{kind}({b.get('source')})"
        return f"{b.get('target')} = {b.get('source')}"
    if cid == "BUILTIN_REDUCE":
        return f"{b.get('op')}({b.get('source')})"
    if cid == "CALL":
        return f"calls {b.get('callee')}(...)"
    return str(b)


def generate_report(source: str, model: CompetencyModel, target_ids: list, function_name: str = None,
                     semantic_backup: bool = False, semantic_threshold: float = 0.5,
                     semantic_backend: str = "stub") -> dict:
    """Build the structured report dict for one submission's source code.

    If function_name is given, both the primitive inventory and the
    competency resolution are scoped to JUST that function's own body --
    this matters because resolving against the whole file only guards
    against evidence from two DIFFERENT unrelated functions combining
    within one competency (the per-function scoping guard); it does
    nothing to stop an unrelated helper function elsewhere in the file
    from satisfying a competency on its own, which then gets credited to
    the file as a whole even though the function actually being graded
    (e.g. `top_k`) might not implement it at all. When you know which
    function the assignment asks students to write, always pass it.

    If semantic_backup is True, any competency whose rule-based
    final_score falls below semantic_threshold additionally gets a
    'semantic_backup' field: a CodeBERT-similarity comparison against
    known-correct exemplars (see semantic_backup.py), reported strictly
    as a separate, unverified signal -- it never changes final_score.
    Use this to flag submissions worth a second look when the structural
    rules find nothing, not as an alternate scoring path.
    """
    tree = ast.parse(source)

    if function_name is not None:
        target_node = next(
            (n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and n.name == function_name),
            None,
        )
        if target_node is None:
            return {"error": f"function '{function_name}' not found in submission",
                    "primitives": {}, "competencies": {}}
        scan_tree = target_node
        results = model.resolve(target_node, target_ids, module_tree=tree)
    else:
        scan_tree = tree
        results = model.resolve(tree, target_ids)

    pidx = _ParentIndex(scan_tree)
    primitives = {}
    for pid, detector in PRIMITIVE_DETECTORS.items():
        evs = detector(scan_tree, pidx)
        if evs:
            primitives[pid] = [
                {"line": ev.start_line, "end_line": ev.end_line, "summary": _describe(ev)}
                for ev in evs
            ]

    snippet_source = ast.get_source_segment(source, scan_tree) if function_name else source

    competencies = {}
    for cid in target_ids:
        if cid not in results:
            competencies[cid] = {"error": "not found / unresolved"}
            continue
        res = results[cid]
        comp_def = model.definitions.get(cid)
        declared_slots = list(comp_def.components.keys()) if comp_def else []
        present_evmap = res.evidences[0].children if res.evidences else {}
        present = [s for s in declared_slots if s in present_evmap]
        missing = [s for s in declared_slots if s not in present_evmap]
        entry = {
            "final_score": round(res.final_score, 2),
            "coverage_score": round(res.coverage_score, 2),
            "structural_score": round(res.structural_score, 2),
            "present_slots": present,
            "missing_slots": missing,
            "detail": res.detail,
        }
        if semantic_backup and entry["final_score"] < semantic_threshold and snippet_source:
            backup = semantic_backup_score(snippet_source, cid, backend=semantic_backend)
            if backup is not None:
                entry["semantic_backup"] = backup
        competencies[cid] = entry

    return {"primitives": primitives, "competencies": competencies}


def format_report_text(report: dict, title: str = "") -> str:
    """Render the structured report as readable text."""
    lines = []
    if title:
        lines.append(f"{'='*60}\n{title}\n{'='*60}")

    lines.append("\nPRIMITIVES DETECTED")
    if not report["primitives"]:
        lines.append("  (none)")
    for pid, evs in report["primitives"].items():
        lines.append(f"  {pid} x{len(evs)}")
        for ev in evs:
            lines.append(f"      line {ev['line']:>3}: {ev['summary']}")

    lines.append("\nADVANCED COMPETENCIES")
    for cid, c in report["competencies"].items():
        if "error" in c:
            lines.append(f"  {cid:20s}  {c['error']}")
            continue
        marker = "OK  " if c["final_score"] >= 0.999 else ("--  " if c["final_score"] == 0 else "~~  ")
        lines.append(f"  {marker}{cid:20s} final={c['final_score']:.2f}  "
                      f"(coverage={c['coverage_score']:.2f} structural={c['structural_score']:.2f})")
        if c["present_slots"]:
            lines.append(f"        present: {', '.join(c['present_slots'])}")
        if c["missing_slots"]:
            lines.append(f"        missing: {', '.join(c['missing_slots'])}")
        if c["coverage_score"] >= 0.999 and c["structural_score"] < 0.999:
            lines.append("        note: all slots had SOME matching candidate, "
                         "but the required arrangement (or specific kind) was not satisfied")
        if "semantic_backup" in c:
            sb = c["semantic_backup"]
            lines.append(f"        semantic backup ({sb['backend']}): {sb['max_similarity']:.2f} similarity "
                         f"to exemplar #{sb['best_match_exemplar_index']} "
                         f"-- UNVERIFIED, flag for manual review, not auto-credited")
    return "\n".join(lines)
'''

with open('report.py', 'w') as f:
    f.write(report_source)

print('Wrote:', 'competency.py', 'test_real_data.py', 'pw01_competencies.py', 'q2_competencies.py', 'report.py')


Wrote: competency.py test_real_data.py pw01_competencies.py q2_competencies.py report.py


## 3. Write the semantic backup module + validation script

In [3]:
semantic_backup_source = r'''










"""
Semantic similarity backup signal.

Structural rules (competency.py) stay the source of truth for final_score --
this module only ever contributes an advisory 'semantic_backup' field on
low-scoring competencies (see report.py), never a grade. Its job is to flag
"this looks like it might be doing the right thing via a strategy no
structural rule recognizes yet" for manual review.

Backends provided:

  - "codet5p": Salesforce/codet5p-110m-embedding via `transformers`.
    Replaces an earlier CodeBERT-based attempt (mean-pooled, then
    [CLS]-pooled) that was measured -- not assumed -- to have no real
    discriminative power: both pooling strategies over bare CodeBERT
    produced SAME/CROSS/UNRELATED similarities bunched in a narrow
    0.96-0.99 band, worse than even the local lexical baseline. That's
    the textbook anisotropic-embedding-space failure of pooling a model
    that was never trained to produce a similarity-meaningful vector in
    the first place. CodeT5+'s 110M embedding checkpoint is trained
    specifically to fix that: its pretraining includes a contrastive
    learning objective over code (paired with span denoising and causal
    LM), producing a purpose-built 256-dim embedding directly from
    `model(input_ids)` -- no manual pooling choice to get right or wrong.
    Needs `torch` + `transformers`, `trust_remote_code=True` (the model
    class ships as custom code in the HF repo, not in the `transformers`
    library itself), and network access to huggingface.co the first time
    (weights are then cached locally).

  - "lexical": a fully local, dependency-light fallback (hashed bag of
    code-token n-grams, cosine similarity). No model download, no network,
    ever. This is deliberately NOT presented as a transformer -- it exists
    so the surrounding plumbing (exemplar registry, caching, threshold
    gating, discriminative-power validation) can be built and tested
    end-to-end in environments without model-hub access, and so
    validate_semantic_backup.py has something to compare "codet5p" against
    once it *can* run.

Both backends implement the same tiny interface (`Embedder.embed(list[str])
-> np.ndarray`), so swapping which one report.py uses is a one-line change
(the `backend=` argument already threaded through generate_report).

Embeddings are cached to disk (keyed by backend name + content hash) since
exemplar sets are re-embedded on every process start otherwise, and
transformer embedding is not cheap.
"""

from __future__ import annotations

import hashlib
import os
import pickle
from typing import Optional

import numpy as np

CACHE_DIR = os.environ.get(
    "SEMANTIC_BACKUP_CACHE_DIR",
    os.path.join(os.path.dirname(os.path.abspath(__file__)), ".semantic_cache"),
)
os.makedirs(CACHE_DIR, exist_ok=True)


# ---------------------------------------------------------------------------
# Exemplar registry
# ---------------------------------------------------------------------------
# Originally 1-2 snippets per competency, seeded just to test wiring (the
# state the methodology doc names as a known limitation). Expanded here
# after validate_semantic_backup.py surfaced two concrete problems with the
# original set:
#
#   1. Six of eight competencies had exactly ONE exemplar, which means they
#      could never be held-out tested at all (nothing to hold them out
#      against) -- the held-out test was silently only exercising 2 of 8
#      competencies. Every competency below now has >=2 exemplars so the
#      validation actually covers the registry it claims to.
#
#   2. Both the lexical AND codet5p backends independently flagged the same
#      confusion: Accumulation's original input()-reading exemplar and
#      GuardedInput's exemplar share surface shape (loop + input() +
#      reassignment) even though they represent different skills
#      (running-sum vs. retry-until-valid). Two very different embedding
#      methods agreeing on the same failure is a signal the EXEMPLARS were
#      ambiguous, not that either embedder is broken. Accumulation now
#      includes a third exemplar that accumulates over an already-available
#      list with no input()/retry semantics at all, and GuardedInput's new
#      second exemplar validates a non-numeric answer with no running total
#      involved -- sharpening the two concepts apart at the exemplar level,
#      which no embedding choice can do on its own.
#
# Snippets are drawn from this project's own synthetic test cases where
# available (test_real_data.py, q2_competencies.py) rather than invented
# fresh; new additions follow the same style (small, single-function,
# representative of one real strategy) rather than being padded filler.

EXEMPLAR_REGISTRY: dict[str, list[str]] = {
    "Swap": [
        """
def f(a, b):
    temp = a
    a = b
    b = temp
    return a, b
""",
        """
def f(a, b):
    a, b = b, a
    return a, b
""",
    ],
    "Accumulation": [
        """
def moyenne(n):
    total = 0
    for i in range(n):
        note = float(input("Note: "))
        total += note
    return total
""",
        """
def total_of(values):
    result = 0
    for v in values:
        result = result + v
    return result
""",
        # New: accumulates over an already-available sequence via a while
        # loop, with no input() call and no retry/validation semantics --
        # deliberately unlike GuardedInput's shape, to stop the two
        # competencies' exemplars from overlapping on "loop + input() +
        # reassignment" alone.
        """
def total_from_list(values):
    total = 0
    i = 0
    while i < len(values):
        total += values[i]
        i += 1
    return total
""",
    ],
    "ConditionalCount": [
        """
def unique_day(day, possible_birthdays):
    count = 0
    for birthday in possible_birthdays:
        if birthday[1] == day:
            count += 1
    return count == 1
""",
        # New: same shape (loop + condition + counter), different domain
        # (character counting rather than tuple-field matching), so the
        # exemplar set isn't tied to one surface vocabulary.
        """
def count_vowels(word):
    count = 0
    for ch in word:
        if ch in "aeiouAEIOU":
            count += 1
    return count
""",
        # New: a third domain (counting negatives), deliberately NOT
        # mirrored by any AccumulateThenCount exemplar below. The
        # unique_day exemplar above shares its exact function name,
        # parameters, and return-comparison wrapper with
        # AccumulateThenCount's unique_day exemplar (confirmed via direct
        # inspection: only the loop body differs) -- real and worth
        # keeping as an authentic hard case, but its near-identical surface
        # text was very likely inflating same-vs-cross similarity for
        # reasons beyond genuine strategy-level equivalence. This exemplar
        # and AccumulateThenCount's third (below) are chosen to NOT form
        # another such minimal pair, so the class average isn't dominated
        # by confounded text similarity.
        """
def count_negatives(values):
    count = 0
    for v in values:
        if v < 0:
            count += 1
    return count
""",
        # Real, mined via mine_exemplars.py from question_2/code/correct
        # (correct_2_010.py), verified final_score=1.00. A genuinely
        # different real idiom for the same match-test: `day in i` (tuple
        # membership) instead of `i[1] == day` (indexed field comparison).
        """
def unique_day(day, possible_birthdays):
    counter = 0
    for i in possible_birthdays:
        if day in i:
            counter += 1
    return counter == 1
""",
        # Real, mined (correct_2_016.py), verified final_score=1.00.
        # Different real idiom again: indexed range() loop instead of
        # direct iteration, and an explicit if/else return instead of a
        # boolean expression -- messier, more typical of real student code
        # than any hand-written exemplar in this registry.
        """
def unique_day(day, possible_birthdays):
    count = 0
    for i in range(len(possible_birthdays)):
        if day == possible_birthdays[i][1]:
            count += 1
    if count == 1:
        return True
    else:
        return False
""",
    ],
    "AccumulateThenCount": [
        # Real, verified: this is one of only 2 (of 291) question_2
        # submissions that structurally satisfy AccumulateThenCount, mined
        # via mine_exemplars.py -- both real matches collapse to this exact
        # shape. AccumulateThenCount is a genuinely rare strategy in this
        # corpus (<1% of correct submissions), not just an
        # under-represented one, and no amount of further mining produces
        # more real structural diversity for it (confirmed independently
        # against the analogous unique_month function -- same 2 students,
        # same shape).
        """
def unique_day(day, possible_birthdays):
    all_days = ()
    for i in possible_birthdays:
        all_days += (i[1],)
    return all_days.count(day) == 1
""",
        # CORRECTED (see semantic_backup_findings.md, finding #7): this
        # used to be `count_evens` (list .append() + len()), which scores
        # final_score=0.00 under the real rule -- AccumulateThenCount
        # specifically requires `.count(` in the output text; `len(...)`
        # doesn't satisfy it. Since real diversity for this competency is
        # exhausted (see above), this is a verified-valid SYNTHETIC
        # replacement: same tuple-accumulate-then-.count() shape as the
        # real exemplar, different domain, confirmed final_score=1.00.
        """
def is_only_occurrence(word, sentence_words):
    seen = ()
    for w in sentence_words:
        seen += (w,)
    return seen.count(word) == 1
""",
        # CORRECTED, same reason: replaces the invalid `count_long_words`
        # (also len()-based, also final_score=0.00). Verified final_score
        # =1.00.
        """
def is_unique_tag(tag, tagged_items):
    matches = ()
    for item in tagged_items:
        matches += (item,)
    return matches.count(tag) == 1
""",
    ],
    "GuardedInput": [
        # CORRECTED (see semantic_backup_findings.md, finding #7): the
        # original two exemplars here BOTH scored final_score=0.0 under the
        # actual structural resolver -- `while n <= 0: n = input(...)` has
        # no explicit `if`, and GuardedInput's rule requires a TEST (an
        # if-statement) nested inside the loop; the while-condition itself
        # doesn't register as TEST evidence. This was never caught because
        # hand-written exemplars were never run through model.resolve() to
        # confirm they actually satisfy the competency they're meant to
        # exemplify -- unlike mined exemplars (mine_exemplars.py), which are
        # structurally verified by construction. Every exemplar below is
        # confirmed final_score=1.00 before being included.
        """
def get_positive():
    n = int(input("Enter a positive number: "))
    valid = False
    while not valid:
        if n > 0:
            valid = True
        else:
            n = int(input("Enter a positive number: "))
    return n
""",
        # Non-numeric domain, no running total anywhere -- isolates the
        # "retry until valid" shape from Accumulation's "keep a running
        # sum" shape, same intent as before, now with a verified structure.
        """
def get_yes_no():
    answer = input("yes or no? ")
    while True:
        if answer in ("yes", "no"):
            break
        answer = input("yes or no? ")
    return answer == "yes"
""",
    ],
    "SelectionStep": [
        """
def find_index(values, target):
    found = -1
    for i, v in enumerate(values):
        if v == target:
            found = i
    return found
""",
        # New: selection by comparison (running minimum) rather than
        # selection by equality match, covering the other real SelectionStep
        # shape (this is also the shape FindExtremum's "manual" branch
        # composes on top of).
        """
def index_of_min(values):
    best = 0
    for i in range(1, len(values)):
        if values[i] < values[best]:
            best = i
    return best
""",
    ],
    "FindExtremumBuiltin": [
        """
def largest(values):
    return max(values)
""",
        # New: min() rather than max() -- same competency, other direction.
        """
def smallest(values):
    return min(values)
""",
    ],
    "TopKPattern": [
        """
def top_k(values, k):
    result = []
    remaining = list(values)
    for _ in range(k):
        best = remaining[0]
        for v in remaining:
            if v > best:
                best = v
        remaining.remove(best)
        result.append(best)
    return result
""",
        # New: same repeated-extraction shape, but removes by INDEX
        # (tracked during the scan, then .pop(idx)) rather than by VALUE
        # (.remove(best) after the scan) -- a real variant seen in student
        # code, and a different enough surface form to test whether the
        # embedding generalizes past one specific coding style.
        """
def bottom_k(values, k):
    pool = list(values)
    result = []
    for _ in range(k):
        idx = 0
        for i in range(1, len(pool)):
            if pool[i] < pool[idx]:
                idx = i
        result.append(pool.pop(idx))
    return result
""",
    ],
    # Real, mined via mine_exemplars.py from question_2/code/correct
    # (unique_day), all verified final_score=1.00. Occurs in 13/291 (4.5%)
    # of correct submissions -- uncommon but not vanishingly rare, unlike
    # AccumulateThenCount -- with 11 distinct structural shapes among those
    # 13. These three were deliberately checked against the specific
    # sub-branches (InlineFunctionalCheck / GuardedFunctionalCheck /
    # InlineTestFunctionalCheck) and confirmed to cover all three real
    # shapes the project's own comments describe, one exemplar each.
    "FunctionalUniqueCheck": [
        # InlineFunctionalCheck shape: everything in one return.
        """
def unique_day(day, possible_birthdays):
    return len(filter(lambda x:x[1]==day, possible_birthdays)) == 1
""",
        # GuardedFunctionalCheck shape: build the collection, then check it.
        """
def unique_day(day, possible_birthdays):
    count = tuple(filter(lambda x: x[1] == day, possible_birthdays))
    if len(count) == 1:
        return True
    return False
""",
        # InlineTestFunctionalCheck shape: pipeline embedded directly in
        # the if-condition itself.
        """
def unique_day(day, possible_birthdays):
    if (tuple(map(lambda x: x[1], possible_birthdays))).count(day) == 1:
        return True
    return False
""",
    ],
    # Real, mined via mine_exemplars.py's mine_cross_function() (needed
    # because DELEGATES reasons across function boundaries -- the whole
    # module tree has to be resolved, not just contains_unique_day in
    # isolation). 227/291 (78%) of question_2 submissions verify at
    # final_score=1.00 -- a common, well-represented real pattern, with 96
    # distinct structural shapes. These three span clean/idiomatic,
    # compact/single-condition, and real messy-but-correct student style.
    # NOTE on scope: these exemplars include BOTH unique_day and
    # contains_unique_day, because DELEGATES needs the callee's own body
    # present in the same parsed tree to verify structurally (see
    # audit_registry -- without unique_day present, resolve() can't find
    # it via function_index and the check trivially scores 0.0).
    #
    # This USED to also be what got embedded for similarity, which was a
    # real bug, not just an asymmetry: report.py's generate_report(), when
    # scoped via function_name, only ever passes the TARGET function's own
    # text as snippet_source (ast.get_source_segment on just that
    # function) -- so a live comparison at runtime embeds only
    # contains_unique_day, never paired with unique_day the way these
    # exemplars are written. Embedding the full two-function text made
    # every DelegatesUniqueness exemplar literally CONTAIN a full
    # ConditionalCount implementation, which is exactly what caused the
    # ConditionalCount/DelegatesUniqueness confound found during
    # validation (a lexical/bag-of-tokens method has no way to tell
    # "contains a counter" apart from "is one").
    #
    # Fixed via DELEGATING_FUNCTION + runtime_scoped_exemplar() below:
    # audit_registry() still verifies the full text here (it must, to
    # resolve DELEGATES), but every embedding call site now scopes
    # DelegatesUniqueness exemplars down to just contains_unique_day's own
    # source before computing similarity -- matching what report.py
    # actually compares against in production. The registry storage
    # didn't need to change, only what gets embedded.
    # FIX (see DELEGATING_FUNCTION / runtime_scoped_exemplar below): kept as
    # full two-function text here because audit_registry() genuinely needs
    # unique_day present to verify DELEGATES. The scope mismatch this used
    # to cause -- embedding this full text, when report.py only ever passes
    # contains_unique_day's own source at runtime -- is now handled at the
    # embedding call sites, not by changing what's stored here.
    "DelegatesUniqueness": [
        """
def unique_day(day, possible_birthdays):
    count = 0
    for birthday in possible_birthdays:
        if birthday[1] == day:
            count += 1
    return count == 1

def contains_unique_day(month, possible_birthdays):
    for birthday in possible_birthdays:
        if birthday[0] == month:
            if unique_day(birthday[1], possible_birthdays):
                return True
    return False
""",
        """
def unique_day(day, possible_birthdays):
    count=0
    for i in possible_birthdays:
        if i[1]==day:
            count+=1
    return count==1

def contains_unique_day(month, possible_birthdays):
    for i in possible_birthdays:
        if i[0]==month and unique_day(i[1],possible_birthdays):
            return True
    return False
""",
        """
def unique_day(day, possible_birthdays):
    days = 0
    for i in range(len(possible_birthdays)):
        if possible_birthdays[i][1] == day:
            days = days + 1
    if days == 1:
        return True
    else:
        return False

def contains_unique_day(month, possible_birthdays):
    tup = ()
    for k in range(len(possible_birthdays)):
        if possible_birthdays[k][0] == month:
            tup = tup + (possible_birthdays[k],)
    for l in range(len(tup)):
        if unique_day(tup[l][1], possible_birthdays):
            return True
    return False
""",
    ],
}


# ---------------------------------------------------------------------------
# Runtime-scope correction for exemplars that need MORE context to verify
# than report.py ever actually embeds for a real submission.
# ---------------------------------------------------------------------------
# Maps competency id -> name of the function report.py actually scopes to
# and embeds at runtime for that competency, when it differs from "the
# whole exemplar text" (the common case, which needs no entry here at
# all). DelegatesUniqueness is the first case that needs this: its
# exemplars must include the delegated-to function (unique_day) for
# audit_registry()/DELEGATES to verify them, but a real submission is only
# ever scored -- and its snippet embedded -- as contains_unique_day alone.
DELEGATING_FUNCTION: dict[str, str] = {
    "DelegatesUniqueness": "contains_unique_day",
}


def runtime_scoped_exemplar(cid: str, source: str) -> str:
    """Return the substring of `source` that report.py would actually pass
    as snippet_source at runtime for competency `cid`.

    For any cid not in DELEGATING_FUNCTION (i.e. almost all of them), this
    is the identity function -- the full exemplar text already matches
    what gets embedded in production. Only competencies whose exemplars
    need MORE context to verify than a real submission is ever scored with
    (currently just DelegatesUniqueness) need scoping down before
    similarity is computed.

    Used at every embedding call site (semantic_backup_score,
    validate_semantic_backup.py's held-out test) -- deliberately NOT used
    by audit_registry(), which needs the full, unscoped text to verify
    DELEGATES structurally in the first place. Falls back to the original
    source (rather than raising) if the named function isn't found or the
    source doesn't parse, so a malformed exemplar fails loudly elsewhere
    (audit_registry) rather than silently here.
    """
    fn_name = DELEGATING_FUNCTION.get(cid)
    if fn_name is None:
        return source
    import ast as _ast
    try:
        tree = _ast.parse(source)
    except SyntaxError:
        return source
    fn_node = next(
        (n for n in _ast.walk(tree)
         if isinstance(n, _ast.FunctionDef) and n.name == fn_name),
        None,
    )
    if fn_node is None:
        return source
    segment = _ast.get_source_segment(source, fn_node)
    return segment if segment is not None else source


def audit_registry(model, registry: dict[str, list[str]] | None = None) -> dict[str, list[str]]:
    """Check every exemplar in `registry` (default: EXEMPLAR_REGISTRY)
    against the ACTUAL structural resolver, using `model` (a
    CompetencyModel with the relevant competencies registered -- e.g. the
    `model` from test_real_data.py).

    This exists because of a real bug: both original GuardedInput
    exemplars scored final_score=0.0 under the real resolver (a bare
    `while` condition doesn't register as TEST evidence; GuardedInput
    requires an explicit nested `if`), and nothing had ever checked that.
    Mined exemplars (mine_exemplars.py) can't have this problem by
    construction -- they're selected FROM submissions the resolver already
    scored 1.0 -- but hand-written ones can drift silently, e.g. after a
    structural rule changes, or if the author's mental model of a
    competency doesn't quite match its registered definition. Run this
    after editing the registry by hand.

    Returns {cid: [warning strings]} for anything that didn't score
    final_score >= 0.999. A cid missing from `model`'s registrations
    (resolve() returns nothing for it) is reported as a warning too,
    rather than silently skipped, since that's just as easy to miss as a
    low score, and it's exactly the case that made AccumulateThenCount
    unverifiable when this check was first added (it's registered in a
    different module than the one available at the time).
    """
    import ast

    reg = registry if registry is not None else EXEMPLAR_REGISTRY
    problems: dict[str, list[str]] = {}
    for cid, exemplars in reg.items():
        for i, src in enumerate(exemplars):
            try:
                results = model.resolve(ast.parse(src), [cid])
            except SyntaxError as e:
                problems.setdefault(cid, []).append(f"exemplar #{i}: does not parse ({e})")
                continue
            res = results.get(cid)
            if res is None:
                problems.setdefault(cid, []).append(
                    f"exemplar #{i}: {cid!r} not registered in the given model -- "
                    "cannot verify, don't assume it's fine"
                )
            elif res.final_score < 0.999:
                problems.setdefault(cid, []).append(
                    f"exemplar #{i}: final_score={res.final_score:.2f}, "
                    "not a real example of this competency"
                )
    return problems


def _normalize_source(src: str) -> str:
    """Strip leading/trailing blank lines so exemplar formatting quirks
    don't dominate similarity for reasons unrelated to code structure."""
    lines = src.splitlines()
    while lines and not lines[0].strip():
        lines.pop(0)
    while lines and not lines[-1].strip():
        lines.pop()
    return "\n".join(lines)


def _content_key(backend: str, text: str) -> str:
    h = hashlib.sha256(_normalize_source(text).encode("utf-8")).hexdigest()
    return f"{backend}:{h}"


def _load_cache(backend: str) -> dict:
    path = os.path.join(CACHE_DIR, f"{backend}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return {}


def _save_cache(backend: str, cache: dict) -> None:
    path = os.path.join(CACHE_DIR, f"{backend}.pkl")
    with open(path, "wb") as f:
        pickle.dump(cache, f)


# ---------------------------------------------------------------------------
# Embedders
# ---------------------------------------------------------------------------

class Embedder:
    name: str

    def embed(self, texts: list[str]) -> np.ndarray:
        """Return an (n, d) L2-normalized embedding matrix, one row per text."""
        raise NotImplementedError


class LexicalEmbedder(Embedder):
    """Local, model-free baseline: hashed bag of code-token n-grams (1-2),
    L2-normalized so dot product == cosine similarity. No network, no
    download, always available -- exists to validate the plumbing and to
    give validate_semantic_backup.py a baseline to compare CodeT5+ against,
    the same way the original stub was meant to (but actually measured)."""

    name = "lexical"

    def __init__(self, n_features: int = 4096):
        from sklearn.feature_extraction.text import HashingVectorizer

        self._vectorizer = HashingVectorizer(
            n_features=n_features,
            token_pattern=r"[A-Za-z_][A-Za-z0-9_]*|[^\sA-Za-z0-9_]",
            ngram_range=(1, 2),
            norm="l2",
            alternate_sign=False,
        )

    def embed(self, texts: list[str]) -> np.ndarray:
        matrix = self._vectorizer.transform(texts)
        return np.asarray(matrix.todense())


class CodeT5PEmbedder(Embedder):
    """Salesforce/codet5p-110m-embedding -- purpose-built code embedding
    model, L2-normalized 256-dim output.

    Unlike CodeBERT (a masked-language-model encoder repurposed for
    similarity by pooling its hidden states after the fact), this
    checkpoint's pretraining explicitly includes a contrastive learning
    objective over code, alongside span denoising and causal LM. The
    embedding comes straight out of `model(input_ids)` -- there's no
    pooling strategy to choose or get wrong, because producing a single
    similarity-meaningful vector is what the model was trained to do.

    Requires `torch` and `transformers`, and `trust_remote_code=True`
    since the model class is shipped as custom code in the HF repo rather
    than built into the `transformers` library. On first use it downloads
    weights from huggingface.co (or uses a local HF cache set via
    HF_HOME). Raises a clear RuntimeError (not a silent fallback) if the
    model can't be loaded, so a blocked-network failure is never mistaken
    for "codet5p says these are dissimilar."
    """

    name = "codet5p"
    MODEL_ID = "Salesforce/codet5p-110m-embedding"

    def __init__(self):
        try:
            import torch
            from transformers import AutoModel, AutoTokenizer
        except ImportError as e:
            raise RuntimeError(
                "CodeT5PEmbedder requires `torch` and `transformers` "
                "(pip install torch transformers)."
            ) from e

        try:
            self._torch = torch
            self._tokenizer = AutoTokenizer.from_pretrained(
                self.MODEL_ID, trust_remote_code=True
            )

            # This checkpoint's custom modeling code (shipped in the HF repo,
            # not in the `transformers` library itself, and last updated
            # ~2 years ago) builds a T5-style encoder stack underneath. Newer
            # `transformers` versions read config.is_decoder /
            # is_encoder_decoder / use_cache unconditionally when
            # constructing that stack; the custom CodeT5pEmbeddingConfig
            # class (a plain PretrainedConfig subclass, not a T5Config) never
            # sets them, because that requirement didn't exist when this
            # repo's code was written. Patching in the correct values for an
            # encoder-only checkpoint (this model has no decoder or KV cache)
            # fixes real forward-compatibility drift -- it isn't a workaround
            # for a different, harder problem.
            from transformers import AutoConfig

            config = AutoConfig.from_pretrained(self.MODEL_ID, trust_remote_code=True)
            for attr, default in (
                ("is_decoder", False),
                ("is_encoder_decoder", False),
                ("use_cache", False),
            ):
                if not hasattr(config, attr):
                    setattr(config, attr, default)

            self._model = AutoModel.from_pretrained(
                self.MODEL_ID, config=config, trust_remote_code=True
            )
            self._model.eval()
        except OSError as e:
            raise RuntimeError(
                f"Could not load {self.MODEL_ID}. This commonly means "
                "huggingface.co is unreachable from this environment (the "
                "exact failure mode this module was built to detect and "
                "report honestly, rather than silently falling back). "
                f"Underlying error: {type(e).__name__}: {e}"
            ) from e
        except Exception as e:
            raise RuntimeError(
                f"Could not load {self.MODEL_ID} for a reason other than "
                "network access -- likely a version-compatibility issue "
                "between this checkpoint's custom remote code and the "
                "installed `transformers` version, not the network-block "
                "failure mode this module primarily guards against. "
                f"Underlying error: {type(e).__name__}: {e}"
            ) from e

    def embed(self, texts: list[str]) -> np.ndarray:
        # Encoded one at a time (not batched) to match the model card's
        # documented usage exactly -- the custom forward() isn't guaranteed
        # to accept an attention_mask kwarg for padded batches, and getting
        # that wrong silently would corrupt every embedding rather than
        # raising, which is worse than the modest speed cost here (exemplar
        # sets are small, and everything is disk-cached afterward anyway).
        torch = self._torch
        vecs = []
        with torch.no_grad():
            for text in texts:
                input_ids = self._tokenizer.encode(
                    text, truncation=True, max_length=512, return_tensors="pt"
                )
                embedding = self._model(input_ids)[0]  # already L2-normalized per model card
                vecs.append(embedding)
            stacked = torch.stack(vecs)
            # Re-normalize defensively so this backend honors the same
            # "L2-normalized, dot product == cosine" contract as the others,
            # rather than trusting an external guarantee silently.
            normed = torch.nn.functional.normalize(stacked, p=2, dim=1)
            return normed.numpy()


_EMBEDDER_CACHE: dict[str, Embedder] = {}


def get_embedder(backend: str) -> Embedder:
    if backend not in _EMBEDDER_CACHE:
        if backend == "lexical":
            _EMBEDDER_CACHE[backend] = LexicalEmbedder()
        elif backend == "codet5p":
            _EMBEDDER_CACHE[backend] = CodeT5PEmbedder()
        elif backend == "stub":
            _EMBEDDER_CACHE[backend] = None  # handled separately below
        else:
            raise ValueError(f"Unknown backend: {backend!r}")
    return _EMBEDDER_CACHE[backend]


def embed_cached(backend: str, texts: list[str]) -> np.ndarray:
    """Embed `texts`, using a persistent on-disk cache keyed by content hash
    so exemplars aren't re-embedded on every process start, and identical
    submissions across a batch run aren't re-embedded either."""
    embedder = get_embedder(backend)
    cache = _load_cache(backend)
    keys = [_content_key(backend, t) for t in texts]
    missing_idx = [i for i, k in enumerate(keys) if k not in cache]
    if missing_idx:
        fresh = embedder.embed([texts[i] for i in missing_idx])
        for i, vec in zip(missing_idx, fresh):
            cache[keys[i]] = vec
        _save_cache(backend, cache)
    return np.stack([cache[k] for k in keys])


# ---------------------------------------------------------------------------
# Public entry point (matches the signature report.py already calls)
# ---------------------------------------------------------------------------

def semantic_backup_score(
    snippet_source: str, cid: str, backend: str = "lexical"
) -> Optional[dict]:
    """Compare `snippet_source` against cid's exemplars under `backend`.

    Returns None if cid has no exemplars registered (nothing to compare
    against) or if `backend == "stub"` is explicitly requested (kept only
    for backward compatibility with callers that haven't opted into a real
    backend yet). Otherwise returns:
        {"backend": ..., "max_similarity": float in [-1, 1],
         "best_match_exemplar_index": int}

    Never raises on a missing/empty exemplar list (returns None instead,
    since "no exemplars" is a normal, expected state for most competencies
    right now, not an error). DOES raise if the backend itself can't be
    constructed (e.g. codet5p with no network/weights) -- see
    CodeT5PEmbedder's docstring for why that's deliberate.
    """
    if backend == "stub":
        return None

    exemplars = EXEMPLAR_REGISTRY.get(cid, [])
    if not exemplars:
        return None

    # Scope down to what report.py actually embeds at runtime (identity
    # for every cid except DelegatesUniqueness right now -- see
    # runtime_scoped_exemplar's docstring). Without this, exemplars that
    # need extra context to VERIFY (e.g. DELEGATES needing its callee
    # present) would also get embedded WITH that extra context, comparing
    # snippet_source against a strictly easier target than any real
    # submission is ever compared against.
    exemplars = [runtime_scoped_exemplar(cid, e) for e in exemplars]

    query_vec = embed_cached(backend, [snippet_source])[0]
    exemplar_vecs = embed_cached(backend, exemplars)

    sims = exemplar_vecs @ query_vec  # rows are L2-normalized -> dot == cosine
    best_idx = int(np.argmax(sims))
    return {
        "backend": backend,
        "max_similarity": float(sims[best_idx]),
        "best_match_exemplar_index": best_idx,
    }










'''

with open('semantic_backup.py', 'w') as f:
    f.write(semantic_backup_source)
print('wrote semantic_backup.py (with DelegatesUniqueness scope-fix)')


wrote semantic_backup.py (with DelegatesUniqueness scope-fix)


In [4]:
validate_source = r'''




"""
Discriminative-power validation for the semantic backup layer.

The methodology doc is explicit that this was the one thing never actually
checked: "the stand-in embedding used to test the surrounding plumbing was
directly shown to have essentially no discriminative power between related
and unrelated code." This script is that check, run properly, using the
same positive-control / negative-control discipline used everywhere else in
this project's validation methodology (Section 2.6 / Section 4).

For every competency with >=2 exemplars, each exemplar is held out in turn
and compared against:
  - SAME:      its own competency's remaining exemplar(s)      -- should be HIGH
  - CROSS:     other competencies' exemplars                   -- should be LOWER
  - UNRELATED: deliberately irrelevant code (string formatting,
               file I/O, etc., sharing no structural pattern
               with anything in the registry)                  -- should be LOWEST

Two summary metrics are reported, deliberately not just one:

  - "worst-case correct ranking": requires the BEST same-competency
    similarity to beat the WORST cross/unrelated similarity, out of every
    comparison made. This is what earlier runs of this script reported as
    a single "correct ranking %" number. It is an extremely strict
    criterion -- with 8 competencies here, each held-out case is compared
    against ~110+ cross/unrelated exemplars, and it only takes ONE outlier
    among those to fail the case. At this exemplar-set size, the metric
    is close to all-or-nothing regardless of embedding quality, which is
    why it stayed pinned at 20% for codet5p even though the underlying
    similarity means clearly separated further than lexical's did. Kept
    here for continuity with earlier runs, not as the primary verdict.

  - "pairwise win rate" (the standard AUC / Mann-Whitney statistic): the
    fraction of ALL (same-pair, other-pair) comparisons where the
    same-competency similarity is higher. 0.5 = no better than chance,
    1.0 = perfect separation. This is the metric actually used for the
    verdict below, because it reflects average-case separation instead of
    being decided by a single worst-case outlier -- the right lens given
    how few exemplars exist per competency right now.

Usage:
    python validate_semantic_backup.py --backend lexical
    python validate_semantic_backup.py --backend codet5p   # needs torch+transformers+network
    python validate_semantic_backup.py --backend all       # both, side by side
"""

from __future__ import annotations

import argparse
import itertools
import json
import statistics

import numpy as np

from semantic_backup import EXEMPLAR_REGISTRY, embed_cached, get_embedder, runtime_scoped_exemplar

UNRELATED_SNIPPETS = [
    """
def greet(name):
    print("Hello, " + name + "!")
""",
    """
def read_config(path):
    with open(path) as f:
        lines = f.readlines()
    return [line.strip() for line in lines]
""",
    """
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distance_to(self, other):
        return ((self.x - other.x) ** 2 + (self.y - other.y) ** 2) ** 0.5
""",
]


def run(backend: str, registry: dict | None = None, label: str = "") -> None:
    registry = registry if registry is not None else EXEMPLAR_REGISTRY
    header = f"backend = {backend!r}" + (f"  registry = {label!r}" if label else "")
    print(f"\n{'='*72}\n{header}\n{'='*72}")
    try:
        get_embedder(backend)  # fail fast with a clear error, before any comparisons
    except RuntimeError as e:
        print(f"  SKIPPED -- backend unavailable: {e}")
        return

    cids = [cid for cid, exs in registry.items() if len(exs) >= 1]
    scoreable_cids = [cid for cid in cids if len(registry[cid]) >= 2]
    # Scope every exemplar down to what report.py would actually embed at
    # runtime before any similarity is computed (identity for every cid
    # except DelegatesUniqueness right now). Without this, exemplars that
    # need extra context to VERIFY (DELEGATES needing its callee present)
    # also get embedded WITH that extra context -- comparing against a
    # strictly easier, unrealistic target no real submission ever produces.
    all_exemplars = {
        cid: [runtime_scoped_exemplar(cid, e) for e in exs]
        for cid, exs in registry.items()
    }

    skipped = [cid for cid in cids if cid not in scoreable_cids]
    if skipped:
        print(f"  (skipping held-out test for {len(skipped)} single-exemplar "
              f"competencies -- nothing to hold them out against: {', '.join(skipped)})")

    same_sims, cross_sims, unrelated_sims = [], [], []
    correct_margin = 0
    total_cases = 0
    win_count = 0
    win_total = 0
    # For diagnosing REAL confusions (e.g. two competencies that are
    # legitimately similar in shape) vs. noise: track, per held-out case,
    # what beat it and by how much.
    worst_case_failures = []

    unrelated_vecs = embed_cached(backend, UNRELATED_SNIPPETS)

    for cid in scoreable_cids:
        exemplars = all_exemplars[cid]
        vecs = embed_cached(backend, exemplars)
        for held_out_idx in range(len(exemplars)):
            query = vecs[held_out_idx]
            others_same = [vecs[i] for i in range(len(exemplars)) if i != held_out_idx]

            same_vals = [float(query @ v) for v in others_same]
            cross_pairs = [
                (other_cid, float(query @ v))
                for other_cid in cids if other_cid != cid
                for v in embed_cached(backend, all_exemplars[other_cid])
            ]
            unrel_vals = [float(query @ v) for v in unrelated_vecs]
            other_vals = [v for _, v in cross_pairs] + unrel_vals

            same_sims.extend(same_vals)
            cross_sims.extend(v for _, v in cross_pairs)
            unrelated_sims.extend(unrel_vals)

            total_cases += 1
            best_same = max(same_vals) if same_vals else None
            worst_other = max(other_vals) if other_vals else None
            if best_same is not None and worst_other is not None:
                if best_same > worst_other:
                    correct_margin += 1
                else:
                    beater_cid = next(
                        (c for c, v in cross_pairs if v == worst_other), "UNRELATED"
                    )
                    worst_case_failures.append((cid, beater_cid, best_same, worst_other))

            # Pairwise win rate: every (same, other) comparison counts
            # independently, so one outlier can't flip an entire case.
            for s in same_vals:
                for o in other_vals:
                    win_total += 1
                    if s > o:
                        win_count += 1

    def summarize(name, vals):
        if not vals:
            print(f"  {name:10s}  (n/a -- no pairs)")
            return
        print(f"  {name:10s}  mean={statistics.mean(vals):+.3f}  "
              f"median={statistics.median(vals):+.3f}  n={len(vals)}")

    print(f"\n  Competencies with exemplars: {len(cids)}  |  held-out cases: {total_cases}\n")
    summarize("SAME", same_sims)
    summarize("CROSS", cross_sims)
    summarize("UNRELATED", unrelated_sims)

    pct = 100.0 * correct_margin / total_cases if total_cases else 0.0
    print(f"\n  Worst-case correct ranking (best SAME > worst of ALL others): "
          f"{correct_margin}/{total_cases} ({pct:.1f}%)  [strict, see docstring]")
    if worst_case_failures:
        print("    Failures (what beat the held-out SAME pair):")
        for cid, beater_cid, best_same, worst_other in worst_case_failures:
            print(f"      {cid:20s} best_same={best_same:+.3f}  "
                  f"beaten by {beater_cid:20s} sim={worst_other:+.3f}")

    win_rate = win_count / win_total if win_total else 0.0
    print(f"\n  Pairwise win rate (SAME beats random OTHER): "
          f"{win_count}/{win_total} ({win_rate:.1%})  [standard AUC-style metric]")
    if win_rate >= 0.85:
        verdict = "strong discriminative power"
    elif win_rate >= 0.65:
        verdict = "real, moderate discriminative power -- better than chance, not yet strong"
    elif win_rate >= 0.55:
        verdict = "weak discriminative power -- only marginally better than chance"
    else:
        verdict = "little to no discriminative power (same failure mode as the original stub)"
    print(f"  Verdict: {verdict}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--backend",
        choices=["lexical", "codet5p", "both", "all"],
        default="all",
    )
    parser.add_argument(
        "--registry-file", default=None,
        help="Path to a JSON exemplar registry (e.g. from mine_exemplars.py's "
             "--out) to validate INSTEAD OF the hand-written EXEMPLAR_REGISTRY "
             "in semantic_backup.py. Use --compare-registries to run both.",
    )
    parser.add_argument(
        "--compare-registries", action="store_true",
        help="With --registry-file set, run BOTH the hand-written and the "
             "mined registry for each backend, back to back, instead of "
             "replacing one with the other. No effect without --registry-file.",
    )
    args = parser.parse_args()

    if args.backend in ("both", "all"):
        backends = ["lexical", "codet5p"]
    else:
        backends = [args.backend]

    mined_registry = None
    if args.registry_file:
        with open(args.registry_file, encoding="utf-8") as f:
            mined_registry = json.load(f)

    for b in backends:
        if mined_registry is None:
            run(b)
        elif args.compare_registries:
            run(b, registry=None, label="hand-written")
            run(b, registry=mined_registry, label="mined")
        else:
            run(b, registry=mined_registry, label="mined")




'''

with open('validate_semantic_backup.py', 'w') as f:
    f.write(validate_source)
print('wrote validate_semantic_backup.py (with DelegatesUniqueness scope-fix)')


wrote validate_semantic_backup.py (with DelegatesUniqueness scope-fix)


## 3b. Audit the exemplar registry against the real structural resolver
Catches the exact class of bug found during this project's own validation: several hand-written exemplars (both original `GuardedInput` ones, two of three `AccumulateThenCount` ones) scored `final_score=0.0` under the real resolver -- nothing had ever checked exemplars against the rules they were supposed to represent. All fixed now, and the registry also includes real, mined, verified exemplars for two more competencies made possible by `pw01_competencies.py`: `FunctionalUniqueCheck` (the third real UniqueCheck strategy -- map()/filter() pipelines) and `DelegatesUniqueness` (cross-function: does `contains_unique_day` correctly delegate to a `unique_day` that itself satisfies `ConditionalCount`, via `DELEGATES`). Uses `q2_competencies.py`'s model (the real chain, not a reconstruction) -- confirmed to produce identical scores to the earlier `q2_extra.py` workaround on 20 real submissions before that workaround was retired.

**Scope-mismatch fix included in this run:** `DelegatesUniqueness`'s exemplars must contain BOTH `unique_day` and `contains_unique_day` for `audit_registry()` to verify them (DELEGATES needs the callee visible in the same parsed tree) -- but `report.py` only ever embeds `contains_unique_day`'s own text at runtime. Embedding the full two-function text used to make every `DelegatesUniqueness` exemplar literally contain a full `ConditionalCount` implementation, which caused a real `ConditionalCount`/`DelegatesUniqueness` confound in an earlier round. `runtime_scoped_exemplar()` now scopes exemplars down to just what `report.py` actually embeds at every similarity call site, while `audit_registry()` below still verifies the full, unscoped text (it must, to resolve `DELEGATES`) -- confirmed on the lexical backend to raise pairwise win rate 88.4% -> 90.6% and worst-case ranking 55.6% -> 74.1%, with the `DelegatesUniqueness` confound eliminated entirely. This run additionally checks whether the same fix changes `codet5p`'s numbers -- not yet confirmed, since codet5p wasn't reachable in the sandbox this fix was made in.


In [5]:
from semantic_backup import audit_registry
import q2_competencies  # registers everything onto test_real_data's shared model,
                         # including pw01_competencies.py's registrations
model = q2_competencies.model

problems = audit_registry(model)
if not problems:
    print("All checkable exemplars verified against the structural resolver.")
else:
    for cid, warnings in problems.items():
        for w in warnings:
            print(f"{cid}: {w}")


All checkable exemplars verified against the structural resolver.


## 4. Run the discriminative-power validation
This downloads `Salesforce/codet5p-110m-embedding` on first run (a couple hundred MB, cached by `transformers` afterward -- note it needs `trust_remote_code=True` since the model class ships as custom code in the HF repo) and compares two backends:
- `lexical` -- local baseline, no model
- `codet5p` -- purpose-built code embedding model, trained with a contrastive objective for code similarity specifically (as opposed to CodeBERT, a masked-language-model encoder repurposed for similarity by pooling after the fact, which measured worse than lexical here).

In [6]:
!python validate_semantic_backup.py --backend all


backend = 'lexical'

  Competencies with exemplars: 10  |  held-out cases: 27

  SAME        mean=+0.643  median=+0.657  n=54
  CROSS       mean=+0.402  median=+0.392  n=648
  UNRELATED   mean=+0.296  median=+0.286  n=81

  Worst-case correct ranking (best SAME > worst of ALL others): 20/27 (74.1%)  [strict, see docstring]
    Failures (what beat the held-out SAME pair):
      Accumulation         best_same=+0.522  beaten by GuardedInput         sim=+0.576
      Accumulation         best_same=+0.391  beaten by ConditionalCount     sim=+0.540
      Accumulation         best_same=+0.522  beaten by SelectionStep        sim=+0.611
      AccumulateThenCount  best_same=+0.657  beaten by FunctionalUniqueCheck sim=+0.700
      GuardedInput         best_same=+0.502  beaten by Accumulation         sim=+0.576
      GuardedInput         best_same=+0.502  beaten by UNRELATED            sim=+0.522
      SelectionStep        best_same=+0.524  beaten by TopKPattern          sim=+0.625

  Pairwise win

## 5. End-to-end integration smoke test
Runs `generate_report()` from your actual `report.py` against a real snippet, with `semantic_backup=True` and `semantic_backend='codet5p'`, to confirm the full pipeline (not just the embedder in isolation).

In [7]:
from report import generate_report, format_report_text
from test_real_data import model
import q2_competencies  # registers DelegatesUniqueness onto the shared model

src = '''
def moyenne(n):
    total = 0
    for i in range(n):
        note = float(input("Note: "))
        total += note
    return total
'''

r = generate_report(
    src, model, ["Accumulation", "TopKPattern"],
    semantic_backup=True, semantic_threshold=0.9, semantic_backend="codet5p",
)
print(format_report_text(r, title="moyenne() -- integration smoke test (codet5p)"))

print()
print('=' * 70)
print('Scope-fix check: DelegatesUniqueness, scoped to contains_unique_day only')
print('=' * 70)
delegation_src = '''
def unique_day(day, possible_birthdays):
    total = 0
    for b in possible_birthdays:
        if b[1] == day:
            total = total + 1
    return total

def contains_unique_day(month, possible_birthdays):
    for b in possible_birthdays:
        if b[0] == month:
            return True
    return False
'''
r2 = generate_report(
    delegation_src, model, ["DelegatesUniqueness"], function_name="contains_unique_day",
    semantic_backup=True, semantic_threshold=1.0, semantic_backend="codet5p",
)
print(format_report_text(r2, title="contains_unique_day() -- DelegatesUniqueness scope-fix check (codet5p)"))


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


moyenne() -- integration smoke test (codet5p)

PRIMITIVES DETECTED
  DECL x1
      line   3: total declared
  INPUT x1
      line   5: input() -> note
  PARAM x1
      line   2: parameter n
  FUNC_DEF x1
      line   2: def moyenne(1 params), returns a value
  OUTPUT x1
      line   7: output: total
  LOOP x1
      line   4: for i in range(n)
  ASSIGN x3
      line   3: total = 0
      line   5: note = float(input('Note: '))
      line   6: total = total Add note

ADVANCED COMPETENCIES
  OK  Accumulation         final=1.00  (coverage=1.00 structural=1.00)
        present: decl, loop, assign
  --  TopKPattern          final=0.00  (coverage=0.33 structural=0.00)
        present: loop
        missing: select, remove
        semantic backup (codet5p): 0.42 similarity to exemplar #1 -- UNVERIFIED, flag for manual review, not auto-credited

Scope-fix check: DelegatesUniqueness, scoped to contains_unique_day only
contains_unique_day() -- DelegatesUniqueness scope-fix check (codet5p)

PRIMITIV

## 5a. Finding 12 follow-up: codet5p on the call-site blind-spot variants
The original probe above (moyenne + one broken contains_unique_day) found codet5p rating a completely broken delegation at 0.97 similarity. Two further variants were tested against lexical only in a prior session (0.78 and 0.81) but never against codet5p. This cell closes that gap: all three variants -- missing call under the original loop shape, missing call under a different loop shape (while+index), and a wrong-function call -- run against codet5p specifically, so the three lexical numbers finally have a codet5p-side comparison.

In [8]:
from report import generate_report, format_report_text
from test_real_data import model
import q2_competencies  # registers DelegatesUniqueness onto the shared model

variants = {
    'original: no call, same loop shape': '''
def unique_day(day, possible_birthdays):
    total = 0
    for b in possible_birthdays:
        if b[1] == day:
            total = total + 1
    return total

def contains_unique_day(month, possible_birthdays):
    for b in possible_birthdays:
        if b[0] == month:
            return True
    return False
''',
    'A: no call, different loop shape (while+index)': '''
def unique_day(day, possible_birthdays):
    total = 0
    for b in possible_birthdays:
        if b[1] == day:
            total = total + 1
    return total

def contains_unique_day(month, possible_birthdays):
    i = 0
    while i < len(possible_birthdays):
        if possible_birthdays[i][0] == month:
            return True
        i = i + 1
    return False
''',
    'B: calls a wrong, irrelevant helper function': '''
def unique_day(day, possible_birthdays):
    total = 0
    for b in possible_birthdays:
        if b[1] == day:
            total = total + 1
    return total

def helper_unrelated(x):
    return x is not None

def contains_unique_day(month, possible_birthdays):
    for b in possible_birthdays:
        if b[0] == month:
            if helper_unrelated(b):
                return True
    return False
''',
}

print(f"{'variant':<48s} {'lexical (prior)':<16s} {'codet5p (this run)'}")
prior_lexical = {
    'original: no call, same loop shape': 0.811,
    'A: no call, different loop shape (while+index)': 0.776,
    'B: calls a wrong, irrelevant helper function': 0.813,
}
for label, src in variants.items():
    r = generate_report(
        src, model, ['DelegatesUniqueness'], function_name='contains_unique_day',
        semantic_backup=True, semantic_threshold=1.0, semantic_backend='codet5p',
    )
    comp = r['competencies']['DelegatesUniqueness']
    sb = comp.get('semantic_backup')
    codet5p_sim = f"{sb['max_similarity']:.3f}" if sb else "n/a (final_score>=threshold)"
    print(f"{label:<48s} {prior_lexical[label]:<16.3f} {codet5p_sim}")


variant                                          lexical (prior)  codet5p (this run)
original: no call, same loop shape               0.811            0.975
A: no call, different loop shape (while+index)   0.776            0.977
B: calls a wrong, irrelevant helper function     0.813            0.961


## 5b. TopKPatternExtended: the embedding-guided clustering work (Section 5.3), independent of the semantic backup layer
This is structural, not semantic-similarity: `TopKViaDelegate` and `SortThenSlice` were built from clustering Q5's 43 unexplained-until-now submissions by codet5p embedding and reading the dominant cluster, then encoded as ordinary `DELEGATES`-based structural rules -- no embedding involved at match time. Included here because it landed in the same round of files as the semantic backup engine updates above; verify it separately from Section 7's results.

The first cell below also reproduces, in miniature, the exact known false positive documented in the paper's Table 5 / Section 8 ("TopKPattern's own combinatorial matcher... found a coincidental combination... without the helper actually implementing the find-max/remove/append pattern" -- 1 of 15 real TopKViaDelegate matches): a `merge_sort`-style helper's own merge loop (`while ...: ... .pop(0)`) coincidentally satisfies `TopKPattern`'s combinatorial matcher on its own, even though merge-sort isn't the find-max/remove/append pattern at all. Confirmed this is pre-existing, not caused by this round's `while`-TEST engine change (reproduces identically under the prior engine).

In [9]:
from test_real_data import model

# Case 1: helper re-implements TopKPattern under another name -- TopKViaDelegate fires.
# (SortThenSlice also fires here, and that's expected, not a bug: a real find-max-remove
# loop always also satisfies HasSubstantialWork's 'has a loop' check. The two branches are
# not meant to be mutually exclusive, just two different sufficient conditions under one ANY.)
delegate_src = '''
def repeated_extraction(lst, k):
    result = []
    for _ in range(k):
        m = max(lst)
        lst.remove(m)
        result.append(m)
    return result

def top_k(lst, k):
    return repeated_extraction(lst, k)
'''

# Case 2: helper is a genuinely different general sort, via PURE recursion (list slicing +
# comprehensions, zero LOOP primitives) -- the exact scenario SELF_RECURSIVE / HasSubstantialWork
# exists for. Only SortThenSlice should fire, cleanly, with no loop for TopKPattern to even
# attempt combinatorial matching against.
quicksort_src = '''
def quicksort(lst):
    if len(lst) <= 1:
        return lst
    pivot = lst[0]
    less = [x for x in lst[1:] if x < pivot]
    more = [x for x in lst[1:] if x >= pivot]
    return quicksort(less) + [pivot] + quicksort(more)

def top_k(lst, k):
    return quicksort(lst)[:k]
'''

# Case 3: the known false positive (Table 5 / Section 8) reproduced directly -- a merge-sort
# helper's OWN merge loop (while + .pop(0)) coincidentally satisfies TopKPattern's combinatorial
# matcher on its own terms, purely by accident of shape, even though merge-sort never implements
# find-max/remove/append. This is why the paper reports 91.9% as 'very likely a few points above
# the true value' rather than reporting it without caveat.
merge_sort_src = '''
def merge_sort(lst):
    if len(lst) <= 1:
        return lst
    mid = len(lst) // 2
    left = merge_sort(lst[:mid])
    right = merge_sort(lst[mid:])
    result = []
    while left and right:
        if left[0] >= right[0]:
            result.append(left.pop(0))
        else:
            result.append(right.pop(0))
    return result + left + right

def top_k(lst, k):
    return merge_sort(lst)[:k]
'''

import ast
cases = [
    ('Case 1: helper re-implements TopKPattern', delegate_src),
    ('Case 2: helper is pure recursion, zero loops (clean SELF_RECURSIVE case)', quicksort_src),
    ('Case 3: helper is merge-sort (reproduces the known Table 5 false positive)', merge_sort_src),
]
for label, src in cases:
    tree = ast.parse(src)
    fn = next(n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and n.name == 'top_k')
    r = model.resolve(fn, ['TopKPattern', 'TopKViaDelegate', 'SortThenSlice', 'TopKPatternExtended'], module_tree=tree)
    print(f'--- {label} ---')
    for cid, res in r.items():
        print(f'  {cid:20s} final={res.final_score:.2f}')
    print()


--- Case 1: helper re-implements TopKPattern ---
  TopKPattern          final=0.00
  TopKViaDelegate      final=1.00
  SortThenSlice        final=1.00
  TopKPatternExtended  final=1.00

--- Case 2: helper is pure recursion, zero loops (clean SELF_RECURSIVE case) ---
  TopKPattern          final=0.00
  TopKViaDelegate      final=0.00
  SortThenSlice        final=1.00
  TopKPatternExtended  final=1.00

--- Case 3: helper is merge-sort (reproduces the known Table 5 false positive) ---
  TopKPattern          final=0.00
  TopKViaDelegate      final=1.00
  SortThenSlice        final=1.00
  TopKPatternExtended  final=1.00



## 6. (Optional) Download the results
Pulls the embedding cache back down so repeat runs in a fresh Colab session don't re-embed everything (and re-download the model) from scratch.

In [10]:
import shutil
shutil.make_archive("semantic_cache", "zip", ".semantic_cache")
files.download("semantic_cache.zip")


NameError: name 'files' is not defined